In [ ]:
# ==============================
# Importación de librerías
# ==============================
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time
import keyboard
import scipy.signal as sig

from PyQt5.QtWidgets import *
from PyQt5.QtCore import *

# ==============================
# Variables controladas por GUI
# ==============================

sensibilidad = 2
mano = 0
fc = 4
ejecutando = False

# ==============================
# Configuración de MediaPipe
# ==============================

mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

window_size = 1000
positions = []

Fs = 30
orden = 1

b, a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Ventana PyQt
# ==============================

class Ventana(QWidget):

    def __init__(self):

        super().__init__()

        self.setWindowTitle("Control Mouse por Gestos")

        layout = QVBoxLayout()

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))

        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")

        self.radio_der.setChecked(True)

        self.radio_der.toggled.connect(self.cambiar_mano)

        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))

        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setMinimum(1)
        self.slider_sens.setMaximum(10)
        self.slider_sens.setValue(2)

        self.slider_sens.valueChanged.connect(self.cambiar_sens)

        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))

        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setMinimum(1)
        self.slider_fc.setMaximum(10)
        self.slider_fc.setValue(4)

        self.slider_fc.valueChanged.connect(self.cambiar_fc)

        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def iniciar_programa(self):

        global ejecutando

        ejecutando = not ejecutando

        if ejecutando:
            self.boton_inicio.setText("Pausar")
        else:
            self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):

        global mano

        if self.radio_der.isChecked():
            mano = 0
        else:
            mano = 1

    def cambiar_sens(self,val):

        global sensibilidad

        sensibilidad = val

    def cambiar_fc(self,val):

        global fc,b,a

        fc = val
        b,a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Filtro suavizado
# ==============================

def suavizar_pos(x, y):

    positions.append((x,y))

    if len(positions) > window_size:
        positions.pop(0)

    xs,ys = zip(*positions)

    x0_filtered = sig.lfilter(b,a,xs)
    y0_filtered = sig.lfilter(b,a,ys)

    return x0_filtered[-1],y0_filtered[-1]

# ==============================
# Scroll
# ==============================

estado_scroll = 0
valor_anterior_y = 0

def mouse_gesto_operacion_scroll(coord_mano):

    global estado_scroll,valor_anterior_y

    scroll = 0

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

            if gesto_estado_subir:
                estado_scroll = 1

            if gesto_estado_bajar:
                estado_scroll = 2

            if gesto_estado_reposo:
                estado_scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 1

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = -1

    else:

        estado_scroll = 0

    return scroll,estado_scroll

# ==============================
# Posición mouse
# ==============================
def mouse_ubicacion(coord_mano):

    mouse_x = mouse_y = 100

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)

            mouse_x,mouse_y = suavizar_pos(mouse_x,mouse_y)

    return mouse_x,mouse_y

# ==============================
# Gestos click
# ==============================
estadoBD2 = estadoBI2 = 0

def mouse_gesto_operacion(coord_mano):

    BI = (0,0)
    BD = (0,0)

    global estadoBD2,estadoBI2

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            BD1 = (hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y)
            BI1 = (hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y)

            BD2 = (hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y)
            BI2 = (hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y)

            BD = (BD1,BD2)
            BI = (BI1,BI2)

    return BD,BI

# ==============================
# Detección mano
# ==============================

def Deteccion_mano_openCV(frame):

    frame = cv2.flip(frame,1)

    frame_rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

    coord_mano = hands.process(frame_rgb)

    return coord_mano

# ==============================
# Detección gestos
# ==============================

def Deteccion_gestos(coord_mano):

    mouse_x,mouse_y = mouse_ubicacion(coord_mano)

    BD,BI = mouse_gesto_operacion(coord_mano)

    return BD,BI,mouse_x,mouse_y

# ==============================
# Clicks mouse
# ==============================

def f_tecla(FF,boton,tecla):

    if FF == 0 and boton[0] == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1

    if FF == 1 and boton[0] == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0

    if boton[1] == 1:
        pyautogui.click(button=tecla,clicks=2)

    return FF

FFD = FFI = 0

# ==============================
# Mouse virtual
# ==============================

def mouse_virtual(x_frame,y_frame,boton_izq,boton_der,scroll,estado_scroll,sensibilidad,mano):

    global FFD,FFI

    pyautogui.scroll(100*scroll)

    width_pantalla,height_pantalla = pyautogui.size()

    y = int((y_frame - height_frame/2)*sensibilidad*height_pantalla/height_frame)

    if mano == 0:
        x = int((x_frame - width_frame/2)*sensibilidad*width_pantalla/width_frame)
    else:
        x = int((x_frame)*sensibilidad*width_pantalla/width_frame)

    if (x>0) and (y>0):

        try:
            pyautogui.moveTo(x,y)
        except pyautogui.FailSafeException:
            pass

    if estado_scroll == 0:

        FFD = f_tecla(FFD,boton_der,'right')
        FFI = f_tecla(FFI,boton_izq,'left')

# ==============================
# Inicializar GUI
# ==============================

app = QApplication(sys.argv)

ventana = Ventana()
ventana.show()

# ==============================
# LOOP PRINCIPAL
# ==============================

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7) as hands:

    while True:

        app.processEvents()

        if not ejecutando:
            continue

        ret,frame = cap.read()

        if ret == False:
            break

        height_frame,width_frame,_ = frame.shape

        coord_mano = Deteccion_mano_openCV(frame)

        BD,BI,mouse_x,mouse_y = Deteccion_gestos(coord_mano)

        scroll,estado_scroll = mouse_gesto_operacion_scroll(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI, scroll,estado_scroll,sensibilidad,mano)

        if keyboard.is_pressed("esc"):            
            break

# ==============================
# Cerrar
# ==============================
cap.release()
cv2.destroyAllWindows()


In [ ]:
# ==============================
# Importación de librerías
# ==============================
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time
import keyboard
import scipy.signal as sig

from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap


# ==============================
# Variables controladas por GUI
# ==============================

sensibilidad = 2
mano = 0
fc = 4
ejecutando = False

# ==============================
# Configuración de MediaPipe
# ==============================

mp_hands = mp.solutions.hands
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

window_size = 1000
positions = []

Fs = 30
orden = 2

b, a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Ventana PyQt
# ==============================

class Ventana(QWidget):

    def __init__(self):

        super().__init__()

        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500,450)

        layout = QVBoxLayout()


        imagen = QLabel()        

        pixmap = QPixmap("mouse virtual logo.jpg")
        
        imagen.setPixmap(pixmap.scaled(200,150))

        imagen.setAlignment(Qt.AlignCenter)
        
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))

        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")

        self.radio_der.setChecked(True)

        self.radio_der.toggled.connect(self.cambiar_mano)

        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))

        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setMinimum(1)
        self.slider_sens.setMaximum(10)
        self.slider_sens.setValue(2)

        self.slider_sens.valueChanged.connect(self.cambiar_sens)

        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))

        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setMinimum(1)
        self.slider_fc.setMaximum(10)
        self.slider_fc.setValue(4)

        self.slider_fc.valueChanged.connect(self.cambiar_fc)

        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def iniciar_programa(self):

        global ejecutando

        ejecutando = not ejecutando

        if ejecutando:
            self.boton_inicio.setText("Pausar")
        else:
            self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):

        global mano

        if self.radio_der.isChecked():
            mano = 0
        else:
            mano = 1

    def cambiar_sens(self,val):

        global sensibilidad
        sensibilidad = val

    def cambiar_fc(self,val):

        global fc,b,a

        fc = val
        b,a = sig.butter(orden, fc/(0.5*Fs), btype='low')

# ==============================
# Filtro suavizado
# ==============================

def suavizar_pos(x, y):

    positions.append((x,y))

    if len(positions) > window_size:
        positions.pop(0)

    xs,ys = zip(*positions)

    x0_filtered = sig.lfilter(b,a,xs)
    y0_filtered = sig.lfilter(b,a,ys)

    return x0_filtered[-1],y0_filtered[-1]

# ==============================
# Scroll
# ==============================

estado_scroll = 0
valor_anterior_y = 0

def mouse_gesto_operacion_scroll(coord_mano):

    global estado_scroll,valor_anterior_y

    scroll = 0

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and estado_scroll == 0
            gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and estado_scroll == 0
            gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

            if gesto_estado_subir:
                estado_scroll = 1

            if gesto_estado_bajar:
                estado_scroll = 2

            if gesto_estado_reposo:
                estado_scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 1

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 1:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (hand_landmarks.landmark[8].y - valor_anterior_y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = 0

            if (valor_anterior_y - hand_landmarks.landmark[8].y)*height_frame > 5 and estado_scroll == 2:
                valor_anterior_y = hand_landmarks.landmark[8].y
                scroll = -1

    else:

        estado_scroll = 0

    return scroll,estado_scroll

# ==============================
# Posición mouse
# ==============================

def mouse_ubicacion(coord_mano):

    mouse_x = mouse_y = 100

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)

            mouse_x,mouse_y = suavizar_pos(mouse_x,mouse_y)

    return mouse_x,mouse_y

# ==============================
# Gestos click
# ==============================

estadoBD2 = estadoBI2 = 0

def mouse_gesto_operacion(coord_mano):

    BI = (0,0)
    BD = (0,0)

    global estadoBD2,estadoBI2

    if coord_mano.multi_hand_landmarks:

        for hand_landmarks in coord_mano.multi_hand_landmarks:

            BD1 = (hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y)
            BI1 = (hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y)

            BD2 = (hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y)
            BI2 = (hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y)

            BD = (BD1,BD2)
            BI = (BI1,BI2)

    return BD,BI

# ==============================
# Detección mano
# ==============================

def Deteccion_mano_openCV(frame):

    frame = cv2.flip(frame,1)
    frame_rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

    coord_mano = hands.process(frame_rgb)

    return coord_mano

# ==============================
# Detección gestos
# ==============================

def Deteccion_gestos(coord_mano):

    mouse_x,mouse_y = mouse_ubicacion(coord_mano)

    BD,BI = mouse_gesto_operacion(coord_mano)

    return BD,BI,mouse_x,mouse_y

# ==============================
# Clicks mouse
# ==============================

def f_tecla(FF,boton,tecla):

    if FF == 0 and boton[0] == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1

    if FF == 1 and boton[0] == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0

    if boton[1] == 1:
        pyautogui.click(button=tecla,clicks=2)

    return FF

FFD = FFI = 0

# ==============================
# Mouse virtual
# ==============================

def mouse_virtual(x_frame,y_frame,boton_izq,boton_der,scroll,estado_scroll,sensibilidad,mano):

    global FFD,FFI

    pyautogui.scroll(100*scroll)

    width_pantalla,height_pantalla = pyautogui.size()

    y = int((y_frame - height_frame/2)*sensibilidad*height_pantalla/height_frame)

    if mano == 0:
        x = int((x_frame - width_frame/2)*sensibilidad*width_pantalla/width_frame)
    else:
        x = int((x_frame)*sensibilidad*width_pantalla/width_frame)

    if (x>0) and (y>0):

        try:
            pyautogui.moveTo(x,y)
        except pyautogui.FailSafeException:
            pass

    if estado_scroll == 0:

        FFD = f_tecla(FFD,boton_der,'right')
        FFI = f_tecla(FFI,boton_izq,'left')

# ==============================
# Inicializar GUI
# ==============================

app = QApplication(sys.argv)

ventana = Ventana()
ventana.show()

# ==============================
# LOOP PRINCIPAL
# ==============================

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7) as hands:

    while True:

        app.processEvents()

        if not ejecutando:
            time.sleep(0.05)   # 🔹 mejora CPU
            continue

        ret,frame = cap.read()

        if ret == False:
            break

        frame = cv2.resize(frame, (640, 480))#

        height_frame,width_frame,_ = frame.shape

        coord_mano = Deteccion_mano_openCV(frame)

        BD,BI,mouse_x,mouse_y = Deteccion_gestos(coord_mano)

        scroll,estado_scroll = mouse_gesto_operacion_scroll(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI, scroll,estado_scroll,sensibilidad,mano)

        if keyboard.is_pressed("esc") or not ventana.isVisible():
            break

# ==============================
# Cerrar
# ==============================

cap.release()
cv2.destroyAllWindows()

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Desactivar el failsafe de pyautogui si prefieres que no crashee en las esquinas
pyautogui.FAILSAFE = False

class VirtualMouse(QWidget):
    def __init__(self):
        super().__init__()
        
        # --- Variables de Control ---
        self.ejecutando = False
        self.sensibilidad = 2
        self.mano_seleccionada = 0 # 0: Derecha, 1: Izquierda
        self.fc = 4
        self.Fs = 30
        self.orden = 1
        self.positions = []
        self.window_size = 5 # Reducido para menor latencia
        
        # Filtro inicial
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')
        
        # MediaPipe Setup
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7,
            model_complexity=0 # 0 es más rápido, 1 es más preciso
        )
        
        self.cap = None
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_frame)
        
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos Pro")
        self.setFixedSize(400, 500)
        layout = QVBoxLayout()

        # Logo / Status
        self.label_status = QLabel("Estado: Detenido")
        self.label_status.setAlignment(Qt.AlignCenter)
        layout.addWidget(self.label_status)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar Cámara")
        self.boton_inicio.setMinimumHeight(50)
        self.boton_inicio.clicked.connect(self.toggle_programa)
        layout.addWidget(self.boton_inicio)

        # Configuración de Mano
        layout.addWidget(QLabel("Seleccionar Mano:"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # Deslizadores
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 15)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        layout.addWidget(QLabel("Suavizado (Frecuencia de Corte)"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    # --- Lógica de Control ---
    def toggle_programa(self):
        if not self.ejecutando:
            self.cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se pudo acceder a la cámara")
                return
            self.ejecutando = True
            self.timer.start(30) # ~30 FPS
            self.boton_inicio.setText("Detener Cámara")
            self.label_status.setText("Estado: Ejecutando")
        else:
            self.ejecutando = False
            self.timer.stop()
            if self.cap:
                self.cap.release()
            self.boton_inicio.setText("Iniciar Cámara")
            self.label_status.setText("Estado: Detenido")

    def cambiar_mano(self):
        self.mano_seleccionada = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        # El filtro lfilter necesita un historial, para pocos datos 
        # a veces es mejor un promedio simple o lfilter con padding
        xf = sig.lfilter(self.b, self.a, xs)[-1]
        yf = sig.lfilter(self.b, self.a, ys)[-1]
        return xf, yf

    def procesar_frame(self):
        ret, frame = self.cap.read()
        if not ret: return

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resultados = self.hands.process(rgb_frame)

        if resultados.multi_hand_landmarks:
            hand_landmarks = resultados.multi_hand_landmarks[0]
            
            # 1. Obtener Posición (Landmark 9 es el centro de la palma, más estable que el 0)
            raw_x = hand_landmarks.landmark[9].x * w
            raw_y = hand_landmarks.landmark[9].y * h
            
            fx, fy = self.suavizar_pos(raw_x, raw_y)
            
            # 2. Mover Mouse
            sw, sh = pyautogui.size()
            # Mapeo con sensibilidad
            nx = int((fx - w/4) * (sw / (w/2)) * (self.sensibilidad/2))
            ny = int((fy - h/4) * (sh / (h/2)) * (self.sensibilidad/2))
            
            try:
                pyautogui.moveTo(nx, ny, _pause=False)
            except: pass

            # 3. Lógica de Clicks (Simplificada)
            # Índice abajo -> Click Izquierdo
            if hand_landmarks.landmark[8].y > hand_landmarks.landmark[6].y:
                pyautogui.click()

    def closeEvent(self, event):
        if self.cap:
            self.cap.release()
        event.accept()

if __name__ == '__main__':
    app = QApplication(sys.argv)
    mouse_app = VirtualMouse()
    mouse_app.show()
    sys.exit(app.exec_())

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # --- Variables de Control ---
        self.ejecutando = False
        self.sensibilidad = 2
        self.mano_seleccionada = 0  # 0: Derecha, 1: Izquierda
        self.fc = 4
        self.Fs = 30
        self.orden = 2
        self.positions = []
        self.window_size = 15
        
        # Variables de estado para clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # --- MediaPipe Setup ---
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        # --- Filtro Inicial ---
        self.actualizar_filtros()

        # --- Timer para el loop (reemplaza al while True) ---
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)
        
        self.cap = cv2.VideoCapture() # Se inicializa vacío

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Imagen/Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.toggle_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(2)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro (Suavizado)"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(4)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    # --- Lógica de Interfaz ---
    def toggle_programa(self):
        if not self.ejecutando:
            # Intentar abrir la cámara solo al iniciar
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                return
            
            self.ejecutando = True
            self.boton_inicio.setText("Pausar / Apagar Cámara")
            self.timer.start(30) # ~30 FPS
            self.positions = [] # Limpiar historial de filtro
        else:
            self.apagar_camara()

    def apagar_camara(self):
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        self.mano_seleccionada = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.actualizar_filtros()

    def actualizar_filtros(self):
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    # --- Procesamiento de Imagen y Gestos ---
    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            xf = sig.lfilter(self.b, self.a, xs)[-1]
            yf = sig.lfilter(self.b, self.a, ys)[-1]
            return xf, yf
        return x, y

    def procesar_loop(self):
        # Verificar tecla de escape
        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret:
            self.apagar_camara()
            return

        frame = cv2.resize(frame, (640, 480))
        h_frame, w_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        res = self.hands.process(frame_rgb)

        if res.multi_hand_landmarks:
            hand_landmarks = res.multi_hand_landmarks[0] # Una sola mano
            
            # 1. Posición del Mouse (Landmark 0)
            mx = int(hand_landmarks.landmark[0].x * w_frame)
            my = int(hand_landmarks.landmark[0].y * h_frame)
            mx, my = self.suavizar_pos(mx, my)

            # 2. Gestos de Click
            # BD = (Anular, Meñique), BI = (Indice, Medio)
            # True si el dedo está "cerrado" (punta por debajo de nudillo)
            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            # 3. Gesto de Scroll
            scroll, est_scroll = self.calcular_scroll(hand_landmarks, w_frame, h_frame)

            # 4. Ejecutar Movimiento Virtual
            self.ejecutar_mouse_virtual(mx, my, (bd1, bd2), (bi1, bi2), 
                                        scroll, est_scroll, w_frame, h_frame)

    def calcular_scroll(self, hand_landmarks, w, h):
        scroll = 0
        # Gesto subir: Indice y Medio pegados horizontalmente
        gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*w < 30) and self.estado_scroll == 0
        # Gesto bajar: Indice muy cerca de su base
        gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*h < 50) and self.estado_scroll == 0
        # Reposo: basado en distancia palma
        gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_subir: self.estado_scroll = 1
        if gesto_bajar: self.estado_scroll = 2
        if gesto_reposo: self.estado_scroll = 0

        # Detección de movimiento vertical del índice para scroll
        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*h > 5:
            if self.estado_scroll == 1:
                scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2:
                scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y

        return scroll, self.estado_scroll

    def ejecutar_mouse_virtual(self, x_f, y_f, boton_der, boton_izq, scroll, est_scroll, w_f, h_f):
        # Scroll
        if scroll != 0:
            pyautogui.scroll(100 * scroll)

        # Movimiento
        sw, sh = pyautogui.size()
        
        # Mapeo de coordenadas
        y = int((y_f - h_f/2) * self.sensibilidad * sh / h_f)
        if self.mano_seleccionada == 0: # Derecha
            x = int((x_f - w_f/2) * self.sensibilidad * sw / w_f)
        else: # Izquierda
            x = int(x_f * self.sensibilidad * sw / w_f)

        if x > 0 and y > 0:
            try:
                pyautogui.moveTo(x, y, _pause=False)
            except:
                pass

        # Clicks (solo si no se está haciendo scroll)
        if est_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, boton_der, 'right')
            self.FFI = self.f_tecla(self.FFI, boton_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        # boton[0] es click simple, boton[1] es doble click
        if FF == 0 and boton[0]:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        elif FF == 1 and not boton[0]:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        
        if boton[1]:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def closeEvent(self, event):
        self.apagar_camara()
        event.accept()

# --- Ejecución ---
if __name__ == "__main__":
    app = QApplication(sys.argv)
    ventana = Ventana()
    ventana.show()
    sys.exit(app.exec_())

In [1]:
#posibale pasaje a james - le falta fs

import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # ==============================
        # Variables controladas por GUI (Tus variables originales)
        # ==============================
        self.sensibilidad = 2
        self.mano = 0
        self.fc = 4
        self.ejecutando = False
        
        # Filtro y posiciones
        self.window_size = 1000
        self.positions = []
        self.Fs = 30
        self.orden = 2
        self.b = None
        self.a = None
        self.actualizar_filtros() # Inicializa b y a

        # Variables de estado para clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # ==============================
        # Configuración de MediaPipe
        # ==============================
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        # Timer para reemplazar el while True y no bloquear la GUI
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)
        
        self.cap = cv2.VideoCapture() 

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    # ==============================
    # Lógica de Interfaz (Tus funciones de Ventana)
    # ==============================
    def iniciar_programa(self):
        self.ejecutando = not self.ejecutando

        if self.ejecutando:
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                self.ejecutando = False
                return
            
            self.boton_inicio.setText("Pausar")
            self.timer.start(30) 
            self.positions = [] 
        else:
            self.apagar_recursos()

    def apagar_recursos(self):
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        if self.radio_der.isChecked():
            self.mano = 0
        else:
            self.mano = 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.actualizar_filtros()

    def actualizar_filtros(self):
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    # ==============================
    # Filtro suavizado
    # ==============================
    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            x0_filtered = sig.lfilter(self.b, self.a, xs)
            y0_filtered = sig.lfilter(self.b, self.a, ys)
            return x0_filtered[-1], y0_filtered[-1]
        return x, y

    # ==============================
    # Loop Principal Corregido
    # ==============================
    def procesar_loop(self):
        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret:
            self.apagar_recursos()
            return

        frame = cv2.resize(frame, (640, 480))
        height_frame, width_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        coord_mano = self.hands.process(frame_rgb)

        if coord_mano.multi_hand_landmarks:
            hand_landmarks = coord_mano.multi_hand_landmarks[0]
            
            # 1. Posición mouse
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x, mouse_y = self.suavizar_pos(mouse_x, mouse_y)

            # 2. Gestos click (BD1, BI1, BD2, BI2)
            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            # 3. Scroll
            scroll, est_scroll = self.mouse_gesto_operacion_scroll(hand_landmarks, width_frame, height_frame)

            # 4. Mouse virtual
            self.mouse_virtual(mouse_x, mouse_y, (bd1, bd2), (bi1, bi2), 
                               scroll, est_scroll, width_frame, height_frame)

    # ==============================
    # Scroll
    # ==============================
    def mouse_gesto_operacion_scroll(self, hand_landmarks, width_frame, height_frame):
        scroll = 0
        gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and self.estado_scroll == 0
        gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and self.estado_scroll == 0
        gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_estado_subir: self.estado_scroll = 1
        if gesto_estado_bajar: self.estado_scroll = 2
        if gesto_estado_reposo: self.estado_scroll = 0

        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*height_frame > 5:
            if self.estado_scroll == 1:
                scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2:
                scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y

        return scroll, self.estado_scroll

    # ==============================
    # Mouse virtual (Lógica de movimiento y Clicks)
    # ==============================
    def mouse_virtual(self, x_frame, y_frame, boton_der, boton_izq, scroll, estado_scroll, width_frame, height_frame):
        if scroll != 0:
            pyautogui.scroll(100 * scroll)

        width_pantalla, height_pantalla = pyautogui.size()
        
        y = int((y_frame - height_frame/2) * self.sensibilidad * height_pantalla / height_frame)
        if self.mano == 0: # Derecha
            x = int((x_frame - width_frame/2) * self.sensibilidad * width_pantalla / width_frame)
        else: # Izquierda
            x = int(x_frame * self.sensibilidad * width_pantalla / width_frame)

        if x > 0 and y > 0:
            try:
                pyautogui.moveTo(x, y, _pause=False)
            except:
                pass

        if estado_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, boton_der, 'right')
            self.FFI = self.f_tecla(self.FFI, boton_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        if FF == 0 and boton[0]:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        elif FF == 1 and not boton[0]:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        
        if boton[1]:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def closeEvent(self, event):
        self.apagar_recursos()
        event.accept()

# --- Ejecución ---
if __name__ == "__main__":
    app = QApplication(sys.argv)
    ventana = Ventana()
    ventana.show()
    app.exec_()

In [2]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # ==============================
        # Variables controladas por GUI
        # ==============================
        self.sensibilidad = 2
        self.mano = 0
        self.fc = 4
        self.ejecutando = False
        
        # Filtro y posiciones
        self.window_size = 1000
        self.positions = []
        self.Fs = 30
        self.orden = 2
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

        # Variables de estado para clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # ==============================
        # Configuración de MediaPipe
        # ==============================
        self.mp_hands = mp.solutions.hands
        # Se inicializa aquí para que persista durante la vida de la ventana
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        self.cap = cv2.VideoCapture() 
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Mouse Virtual")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def iniciar_programa(self):
        self.ejecutando = not self.ejecutando

        if self.ejecutando:
            # CAP_DSHOW es más estable en Windows para evitar que la cámara tarde en encender
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                self.ejecutando = False
                return
            
            self.boton_inicio.setText("Pausar")
            self.timer.start(30) 
            self.positions = [] 
        else:
            self.apagar_recursos()

    def apagar_recursos(self):
        """Libera la cámara y detiene el timer sin cerrar la app"""
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        self.mano = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5*self.Fs), btype='low')

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size:
            self.positions.pop(0)
        
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            x0_filtered = sig.lfilter(self.b, self.a, xs)
            y0_filtered = sig.lfilter(self.b, self.a, ys)
            return x0_filtered[-1], y0_filtered[-1]
        return x, y

    def procesar_loop(self):
        # Esc para cerrar rápido
        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret:
            return

        frame = cv2.resize(frame, (640, 480))
        height_frame, width_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        coord_mano = self.hands.process(frame_rgb)

        if coord_mano.multi_hand_landmarks:
            hand_landmarks = coord_mano.multi_hand_landmarks[0]
            
            # 1. Posición mouse (Usando Landmark 0)
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame)
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x, mouse_y = self.suavizar_pos(mouse_x, mouse_y)

            # 2. Gestos click
            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            # 3. Scroll
            scroll, est_scroll = self.mouse_gesto_operacion_scroll(hand_landmarks, width_frame, height_frame)

            # 4. Mouse virtual
            self.mouse_virtual(mouse_x, mouse_y, (bd1, bd2), (bi1, bi2), 
                               scroll, est_scroll, width_frame, height_frame)

    def mouse_gesto_operacion_scroll(self, hand_landmarks, width_frame, height_frame):
        scroll = 0
        gesto_estado_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and self.estado_scroll == 0
        gesto_estado_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and self.estado_scroll == 0
        gesto_estado_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_estado_subir: self.estado_scroll = 1
        if gesto_estado_bajar: self.estado_scroll = 2
        if gesto_estado_reposo: self.estado_scroll = 0

        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*height_frame > 5:
            if self.estado_scroll == 1:
                scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2:
                scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y

        return scroll, self.estado_scroll

    def mouse_virtual(self, x_frame, y_frame, boton_der, boton_izq, scroll, estado_scroll, width_frame, height_frame):
        if scroll != 0:
            pyautogui.scroll(100 * scroll)

        width_pantalla, height_pantalla = pyautogui.size()
        
        y = int((y_frame - height_frame/2) * self.sensibilidad * height_pantalla / height_frame)
        if self.mano == 0:
            x = int((x_frame - width_frame/2) * self.sensibilidad * width_pantalla / width_frame)
        else:
            x = int(x_frame * self.sensibilidad * width_pantalla / width_frame)

        if x > 0 and y > 0:
            try:
                pyautogui.moveTo(x, y, _pause=False)
            except:
                pass

        if estado_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, boton_der, 'right')
            self.FFI = self.f_tecla(self.FFI, boton_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        if FF == 0 and boton[0]:
            pyautogui.mouseDown(button=tecla)
            FF = 1
        elif FF == 1 and not boton[0]:
            pyautogui.mouseUp(button=tecla)
            FF = 0
        
        if boton[1]:
            pyautogui.click(button=tecla, clicks=2)
        return FF

    def closeEvent(self, event):
        """Este método se ejecuta al cerrar la ventana (X)"""
        self.apagar_recursos()
        # IMPORTANTE: Liberar MediaPipe explícitamente para Jupyter
        if hasattr(self, 'hands'):
            self.hands.close()
        cv2.destroyAllWindows()
        event.accept()

# ==============================
# Ejecución segura para Jupyter
# ==============================
if __name__ == "__main__":
    # Si ya existe una instancia de app, la usamos (evita crasheos en Jupyter)
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    
    try:
        ventana = Ventana()
        ventana.show()
        app.exec_()
    finally:
        # Esto se ejecuta siempre, incluso si hay un error o detienes el Kernel
        cv2.destroyAllWindows()

In [1]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==========================================
# CONFIGURACIÓN Y VARIABLES GLOBALES
# ==========================================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
window_size_filtro = 1000
positions = []
Fs = 30.0
orden = 2
b, a = None, None

# Variables de FPS
tiempo_previo = time.time()
tiempos_fps = []
ventana_promedio = 100 

# Variables de clicks y scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0

# MediaPipe y Cámara
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==========================================
# FUNCIONES DE LÓGICA
# ==========================================

def actualizar_coeficientes_filtro():
    global b, a
    fs_segura = max(Fs, fc * 2.1) 
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')

def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro:
        positions.pop(0)
    
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        x0_filt = sig.lfilter(b, a, xs)
        y0_filt = sig.lfilter(b, a, ys)
        return x0_filt[-1], y0_filt[-1]
    return x, y

def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
    return FF_val

def mouse_gesto_operacion_scroll(hand_landmarks, w_f, h_f):
    global estado_scroll, valor_anterior_y
    scroll = 0
    gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*w_f < 30) and estado_scroll == 0
    gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*h_f < 50) and estado_scroll == 0
    gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

    if gesto_subir: estado_scroll = 1
    if gesto_bajar: estado_scroll = 2
    if gesto_reposo: estado_scroll = 0

    curr_y = hand_landmarks.landmark[8].y
    if abs(curr_y - valor_anterior_y)*h_f > 5:
        if estado_scroll == 1: scroll = 1 if curr_y > valor_anterior_y else 0
        elif estado_scroll == 2: scroll = 0 if curr_y > valor_anterior_y else -1
        valor_anterior_y = curr_y
    return scroll, estado_scroll

def procesar_loop():
    global tiempo_previo, Fs, FFD, FFI
    
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual

    if dt > 0: tiempos_fps.append(dt)
    if len(tiempos_fps) > ventana_promedio: tiempos_fps.pop(0)

    if len(tiempos_fps) == ventana_promedio:
        Fs = 1 / (sum(tiempos_fps) / len(tiempos_fps))
        actualizar_coeficientes_filtro()
        print(f"FPS: {Fs:.2f}", end="\r")

    if keyboard.is_pressed("esc"):
        ventana.close()
        return

    ret, frame = cap.read()
    if not ret: return

    frame = cv2.resize(frame, (640, 480))
    h_frame, w_frame, _ = frame.shape
    frame = cv2.flip(frame, 1)
    res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if res.multi_hand_landmarks:
        hl = res.multi_hand_landmarks[0]
        mx, my = suavizar_pos(int(hl.landmark[0].x * w_frame), int(hl.landmark[0].y * h_frame))

        # Dedos
        bi1, bd1 = hl.landmark[8].y > hl.landmark[5].y, hl.landmark[12].y > hl.landmark[9].y
        bi2, bd2 = hl.landmark[16].y > hl.landmark[13].y, hl.landmark[20].y > hl.landmark[17].y
        
        scroll, est_scroll = mouse_gesto_operacion_scroll(hl, w_frame, h_frame)
        
        # Mouse Virtual
        if scroll != 0: pyautogui.scroll(100 * scroll)
        sw, sh = pyautogui.size()
        y_px = int((my - h_frame/2) * sensibilidad * sh / h_frame)
        x_px = int((mx - w_frame/2) * sensibilidad * sw / w_frame) if mano == 0 else int(mx * sensibilidad * sw / w_frame)

        if x_px > 0 and y_px > 0:
            try: pyautogui.moveTo(x_px, y_px, _pause=False)
            except: pass

        if est_scroll == 0:
            FFD = f_tecla(FFD, (bd1, bd2), 'right')
            FFI = f_tecla(FFI, (bi1, bi2), 'left')

# ==========================================
# FUNCIONES DE INTERFAZ (UI)
# ==========================================

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    ejecutando = not ejecutando
    if ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            QMessageBox.critical(None, "Error", "No hay cámara")
            ejecutando = False
            return
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): cap.release()
    boton_inicio.setText("Iniciar")

def al_cerrar(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==========================================
# SETUP DE LA APLICACIÓN
# ==========================================
actualizar_coeficientes_filtro()

app = QApplication(sys.argv)
ventana = QWidget()
ventana.setWindowTitle("Control Mouse por Gestos")
ventana.setFixedSize(500, 450)
ventana.closeEvent = al_cerrar
layout = QVBoxLayout()

# UI Elements
label_logo = QLabel()
px = QPixmap("mouse virtual logo.jpg")
if not px.isNull(): label_logo.setPixmap(px.scaled(200, 150, Qt.KeepAspectRatio))
label_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.setMinimumHeight(40)
boton_inicio.clicked.connect(iniciar_programa)

radio_der = QRadioButton("Derecha")
radio_izq = QRadioButton("Izquierda")
radio_der.setChecked(True)

def cambio_mano(): global mano; mano = 0 if radio_der.isChecked() else 1
radio_der.toggled.connect(cambio_mano)

slider_sens = QSlider(Qt.Horizontal)
slider_sens.setRange(1, 10)
slider_sens.setValue(sensibilidad)
slider_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

slider_fc = QSlider(Qt.Horizontal)
slider_fc.setRange(1, 10)
slider_fc.setValue(fc)
def cambio_fc(v): global fc; fc = v; actualizar_coeficientes_filtro()
slider_fc.valueChanged.connect(cambio_fc)

# Agregar al layout
layout.addWidget(label_logo)
layout.addWidget(boton_inicio)
layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)
layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(slider_sens)
layout.addWidget(QLabel("Frecuencia corte filtro"))
layout.addWidget(slider_fc)

ventana.setLayout(layout)
timer.timeout.connect(procesar_loop)

ventana.show()
app.exec_()

0

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# Configuración de PyAutoGUI
pyautogui.FAILSAFE = False

class Ventana(QWidget):
    def __init__(self):
        super().__init__()

        # ==============================
        # Variables de Control
        # ==============================
        self.sensibilidad = 2
        self.mano = 0
        self.fc = 4
        self.ejecutando = False
        
        # Filtro y posiciones
        self.window_size_filtro = 1000
        self.positions = []
        self.Fs = 30  # Valor inicial por defecto
        self.orden = 2
        
        # Inicializamos los coeficientes del filtro
        self.actualizar_coeficientes_filtro()

        # Variables para FPS Estables
        self.tiempo_previo = time.time()
        self.tiempos_fps = []
        self.ventana_promedio = 20 
        
        # Variables de clicks y scroll
        self.FFD = 0
        self.FFI = 0
        self.estado_scroll = 0
        self.valor_anterior_y = 0
        
        # ==============================
        # Configuración de MediaPipe
        # ==============================
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        
        self.cap = cv2.VideoCapture() 
        self.timer = QTimer()
        self.timer.timeout.connect(self.procesar_loop)

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("Control Mouse por Gestos")
        self.setFixedSize(500, 450)
        layout = QVBoxLayout()

        # Logo
        imagen = QLabel()
        pixmap = QPixmap("mouse virtual logo.jpg")
        if not pixmap.isNull():
            imagen.setPixmap(pixmap.scaled(200, 150, Qt.KeepAspectRatio))
        imagen.setAlignment(Qt.AlignCenter)
        layout.addWidget(imagen)

        # BOTON INICIAR
        self.boton_inicio = QPushButton("Iniciar")
        self.boton_inicio.setMinimumHeight(40)
        self.boton_inicio.clicked.connect(self.iniciar_programa)
        layout.addWidget(self.boton_inicio)

        # MANO
        layout.addWidget(QLabel("Mano"))
        self.radio_der = QRadioButton("Derecha")
        self.radio_izq = QRadioButton("Izquierda")
        self.radio_der.setChecked(True)
        self.radio_der.toggled.connect(self.cambiar_mano)
        layout.addWidget(self.radio_der)
        layout.addWidget(self.radio_izq)

        # SENSIBILIDAD
        layout.addWidget(QLabel("Sensibilidad"))
        self.slider_sens = QSlider(Qt.Horizontal)
        self.slider_sens.setRange(1, 10)
        self.slider_sens.setValue(self.sensibilidad)
        self.slider_sens.valueChanged.connect(self.cambiar_sens)
        layout.addWidget(self.slider_sens)

        # FRECUENCIA CORTE
        layout.addWidget(QLabel("Frecuencia corte filtro"))
        self.slider_fc = QSlider(Qt.Horizontal)
        self.slider_fc.setRange(1, 10)
        self.slider_fc.setValue(self.fc)
        self.slider_fc.valueChanged.connect(self.cambiar_fc)
        layout.addWidget(self.slider_fc)

        self.setLayout(layout)

    def actualizar_coeficientes_filtro(self):
        # Evitamos que Fs sea demasiado baja para el cálculo
        fs_segura = max(self.Fs, self.fc * 2.1) 
        self.b, self.a = sig.butter(self.orden, self.fc/(0.5 * fs_segura), btype='low')

    def iniciar_programa(self):
        self.ejecutando = not self.ejecutando
        if self.ejecutando:
            self.cap.open(0, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                QMessageBox.critical(self, "Error", "No se puede acceder a la cámara")
                self.ejecutando = False
                return
            self.boton_inicio.setText("Pausar")
            self.timer.start(30) 
            self.positions = [] 
            self.tiempos_fps = [] 
            self.tiempo_previo = time.time()
        else:
            self.apagar_recursos()

    def apagar_recursos(self):
        self.ejecutando = False
        self.timer.stop()
        if self.cap.isOpened():
            self.cap.release()
        self.boton_inicio.setText("Iniciar")

    def cambiar_mano(self):
        self.mano = 0 if self.radio_der.isChecked() else 1

    def cambiar_sens(self, val):
        self.sensibilidad = val

    def cambiar_fc(self, val):
        self.fc = val
        self.actualizar_coeficientes_filtro()

    def suavizar_pos(self, x, y):
        self.positions.append((x, y))
        if len(self.positions) > self.window_size_filtro:
            self.positions.pop(0)
        xs, ys = zip(*self.positions)
        if len(xs) > self.orden * 2:
            x0_filtered = sig.lfilter(self.b, self.a, xs)
            y0_filtered = sig.lfilter(self.b, self.a, ys)
            return x0_filtered[-1], y0_filtered[-1]
        return x, y

    def procesar_loop(self):
        # --- Cálculo de FPS Promediados ---
        tiempo_actual = time.time()
        delta_t = tiempo_actual - self.tiempo_previo
        self.tiempo_previo = tiempo_actual

        if delta_t > 0:
            self.tiempos_fps.append(delta_t)
        if len(self.tiempos_fps) > self.ventana_promedio:
            self.tiempos_fps.pop(0)

        if len(self.tiempos_fps) > 0:
            fps_promedio = 1 / (sum(self.tiempos_fps) / len(self.tiempos_fps))
            
            # Actualizamos self.Fs dinámicamente cada 20 cuadros para que el filtro se adapte
            if len(self.tiempos_fps) == self.ventana_promedio:
                self.Fs = fps_promedio
                self.actualizar_coeficientes_filtro()
            
            print(f"FPS Promedio: {fps_promedio:.2f} | Fs Filtro: {self.Fs:.2f} ", end="\r")

        if keyboard.is_pressed("esc"):
            self.close()
            return

        ret, frame = self.cap.read()
        if not ret: return

        frame = cv2.resize(frame, (640, 480))
        h_frame, w_frame, _ = frame.shape
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        coord_mano = self.hands.process(frame_rgb)

        if coord_mano.multi_hand_landmarks:
            hand_landmarks = coord_mano.multi_hand_landmarks[0]
            mx = int(hand_landmarks.landmark[0].x * w_frame)
            my = int(hand_landmarks.landmark[0].y * h_frame)
            mx, my = self.suavizar_pos(mx, my)

            bi1 = hand_landmarks.landmark[8].y > hand_landmarks.landmark[5].y
            bd1 = hand_landmarks.landmark[12].y > hand_landmarks.landmark[9].y
            bi2 = hand_landmarks.landmark[16].y > hand_landmarks.landmark[13].y
            bd2 = hand_landmarks.landmark[20].y > hand_landmarks.landmark[17].y
            
            scroll, est_scroll = self.mouse_gesto_operacion_scroll(hand_landmarks, w_frame, h_frame)
            self.mouse_virtual(mx, my, (bd1, bd2), (bi1, bi2), scroll, est_scroll, w_frame, h_frame)

    def mouse_gesto_operacion_scroll(self, hand_landmarks, width_frame, height_frame):
        scroll = 0
        gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*width_frame < 30) and self.estado_scroll == 0
        gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*height_frame < 50) and self.estado_scroll == 0
        gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

        if gesto_subir: self.estado_scroll = 1
        if gesto_bajar: self.estado_scroll = 2
        if gesto_reposo: self.estado_scroll = 0

        curr_y = hand_landmarks.landmark[8].y
        if abs(curr_y - self.valor_anterior_y)*height_frame > 5:
            if self.estado_scroll == 1: scroll = 1 if curr_y > self.valor_anterior_y else 0
            elif self.estado_scroll == 2: scroll = 0 if curr_y > self.valor_anterior_y else -1
            self.valor_anterior_y = curr_y
        return scroll, self.estado_scroll

    def mouse_virtual(self, x_f, y_f, b_der, b_izq, scroll, est_scroll, w_f, h_f):
        if scroll != 0: pyautogui.scroll(100 * scroll)
        sw, sh = pyautogui.size()
        y = int((y_f - h_f/2) * self.sensibilidad * sh / h_f)
        x = int((x_f - w_f/2) * self.sensibilidad * sw / w_f) if self.mano == 0 else int(x_f * self.sensibilidad * sw / w_f)

        if x > 0 and y > 0:
            try: pyautogui.moveTo(x, y, _pause=False)
            except: pass

        if est_scroll == 0:
            self.FFD = self.f_tecla(self.FFD, b_der, 'right')
            self.FFI = self.f_tecla(self.FFI, b_izq, 'left')

    def f_tecla(self, FF, boton, tecla):
        if FF == 0 and boton[0]: 
            pyautogui.mouseDown(button=tecla) 
            FF = 1
            
        elif FF == 1 and not boton[0]: 
            pyautogui.mouseUp(button=tecla)
            FF = 0
            
        if boton[1]: 
            pyautogui.click(button=tecla, clicks=2)
            
        return FF

    def closeEvent(self, event):
        self.apagar_recursos()
        if hasattr(self, 'hands'): self.hands.close()
        cv2.destroyAllWindows()
        event.accept()

if __name__ == "__main__":
    app = QApplication.instance() or QApplication(sys.argv)
    try:
        ventana = Ventana()
        ventana.show()
        app.exec_()
    finally:
        cv2.destroyAllWindows()

In [ ]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==========================================
# CONFIGURACIÓN Y VARIABLES GLOBALES
# ==========================================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
window_size_filtro = 1000
positions = []
Fs = 30.0
orden = 2
b, a = None, None

# Variables de FPS
tiempo_previo = time.time()
tiempos_fps = []
ventana_promedio = 50 

# Variables de clicks y scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0

# MediaPipe y Cámara
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==========================================
# FUNCIONES DE LÓGICA
# ==========================================

def actualizar_coeficientes_filtro():
    global b, a
    fs_segura = max(Fs, fc * 2.1) 
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')

def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro:
        positions.pop(0)
    
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        x0_filt = sig.lfilter(b, a, xs)
        y0_filt = sig.lfilter(b, a, ys)
        return x0_filt[-1], y0_filt[-1]
    return x, y

def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
    return FF_val

def mouse_gesto_operacion_scroll(hand_landmarks, w_f, h_f):
    global estado_scroll, valor_anterior_y
    scroll = 0
    gesto_subir = (abs(hand_landmarks.landmark[8].x - hand_landmarks.landmark[13].x)*w_f < 30) and estado_scroll == 0
    gesto_bajar = (abs(hand_landmarks.landmark[5].y - hand_landmarks.landmark[8].y)*h_f < 50) and estado_scroll == 0
    gesto_reposo = (abs(hand_landmarks.landmark[0].y - hand_landmarks.landmark[9].y)/5) < abs(hand_landmarks.landmark[17].x - hand_landmarks.landmark[5].x)

    if gesto_subir: estado_scroll = 1
    if gesto_bajar: estado_scroll = 2
    if gesto_reposo: estado_scroll = 0

    curr_y = hand_landmarks.landmark[8].y
    if abs(curr_y - valor_anterior_y)*h_f > 5:
        if estado_scroll == 1: scroll = 1 if curr_y > valor_anterior_y else 0
        elif estado_scroll == 2: scroll = 0 if curr_y > valor_anterior_y else -1
        valor_anterior_y = curr_y
    return scroll, estado_scroll

def procesar_loop():
    global tiempo_previo, Fs, FFD, FFI
    
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual

    if dt > 0: tiempos_fps.append(dt)
    if len(tiempos_fps) > ventana_promedio: tiempos_fps.pop(0)

    if len(tiempos_fps) == ventana_promedio:
        Fs = 1 / (sum(tiempos_fps) / len(tiempos_fps))
        actualizar_coeficientes_filtro()
        print(f"FPS: {Fs:.2f}", end="\r")

    if keyboard.is_pressed("esc"):
        ventana.close()
        return

    ret, frame = cap.read()
    if not ret: return

    frame = cv2.resize(frame, (640, 480))
    h_frame, w_frame, _ = frame.shape
    frame = cv2.flip(frame, 1)
    res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if res.multi_hand_landmarks:
        hl = res.multi_hand_landmarks[0]
        mx, my = suavizar_pos(int(hl.landmark[0].x * w_frame), int(hl.landmark[0].y * h_frame))

        # Dedos
        bi1, bd1 = hl.landmark[8].y > hl.landmark[5].y, hl.landmark[12].y > hl.landmark[9].y
        bi2, bd2 = hl.landmark[16].y > hl.landmark[13].y, hl.landmark[20].y > hl.landmark[17].y
        
        scroll, est_scroll = mouse_gesto_operacion_scroll(hl, w_frame, h_frame)
        
        # Mouse Virtual
        if scroll != 0: pyautogui.scroll(100 * scroll)
        sw, sh = pyautogui.size()
        y_px = int((my - h_frame/2) * sensibilidad * sh / h_frame)
        x_px = int((mx - w_frame/2) * sensibilidad * sw / w_frame) if mano == 0 else int(mx * sensibilidad * sw / w_frame)

        if x_px > 0 and y_px > 0:
            try: pyautogui.moveTo(x_px, y_px, _pause=False)
            except: pass

        if est_scroll == 0:
            FFD = f_tecla(FFD, (bd1, bd2), 'right')
            FFI = f_tecla(FFI, (bi1, bi2), 'left')

# ==========================================
# FUNCIONES DE INTERFAZ (UI)
# ==========================================

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    ejecutando = not ejecutando
    if ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            QMessageBox.critical(None, "Error", "No hay cámara")
            ejecutando = False
            return
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): cap.release()
    boton_inicio.setText("Iniciar")

def al_cerrar(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==========================================
# SETUP DE LA APLICACIÓN
# ==========================================
actualizar_coeficientes_filtro()

app = QApplication(sys.argv)
ventana = QWidget()
ventana.setWindowTitle("Control Mouse por Gestos")
ventana.setFixedSize(500, 450)
ventana.closeEvent = al_cerrar
layout = QVBoxLayout()

# UI Elements
label_logo = QLabel()
px = QPixmap("mouse virtual logo.jpg")
if not px.isNull(): label_logo.setPixmap(px.scaled(200, 150, Qt.KeepAspectRatio))
label_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.setMinimumHeight(40)
boton_inicio.clicked.connect(iniciar_programa)

radio_der = QRadioButton("Derecha")
radio_izq = QRadioButton("Izquierda")
radio_der.setChecked(True)

def cambio_mano(): global mano; mano = 0 if radio_der.isChecked() else 1
radio_der.toggled.connect(cambio_mano)

slider_sens = QSlider(Qt.Horizontal)
slider_sens.setRange(1, 10)
slider_sens.setValue(sensibilidad)
slider_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

slider_fc = QSlider(Qt.Horizontal)
slider_fc.setRange(1, 10)
slider_fc.setValue(fc)
def cambio_fc(v): global fc; fc = v; actualizar_coeficientes_filtro()
slider_fc.valueChanged.connect(cambio_fc)

# Agregar al layout
layout.addWidget(label_logo)
layout.addWidget(boton_inicio)
layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)
layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(slider_sens)
layout.addWidget(QLabel("Frecuencia corte filtro"))
layout.addWidget(slider_fc)

ventana.setLayout(layout)
timer.timeout.connect(procesar_loop)

ventana.show()
(app.exec_())

In [12]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False

# Variables de Control
sensibilidad = 2
mano = 0
fc = 4
ejecutando = False
positions = []
window_size_filtro = 1000
Fs = 30.0
orden = 2
b, a = None, None

# Estado de Clicks y Scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1, 
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================
def actualizar_coeficientes():
    global b, a
    fs_segura = max(Fs, fc * 2.1)
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')


def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro: 
        positions.pop(0)
        
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        return sig.lfilter(b, a, xs)[-1], sig.lfilter(b, a, ys)[-1]
    return x, y


def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
        
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
        
    return FF_val


def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y

def mouse_gesto_operacion(hl):
    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y
    
    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    return (bi1, bi2),(bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return coord_mano


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


FFD=FFI=0
def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
   # Mouse Move

    global  FFD,FFI
    sw, sh = pyautogui.size()
    y = int((mouse_Y - 240) * sensibilidad * sh / 480)

    if mano == 0:
        x = int((mouse_X - 320) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')


estado_scroll = 0
valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    
    global estado_scroll, valor_anterior_y
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 30) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 50) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    # ESTADOS
    if(gesto_estado_subir): estado_scroll = 1
    if(gesto_estado_bajar): estado_scroll = 2
    if(gesto_estado_reposo): estado_scroll = 0


    # SALIDAS
    #SALIDA SUBIDA
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        #print("1")
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5  and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0


    #SALIDA BAJADA    
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll,estado_scroll


def frecuencia_muestreo():
    global tiempo_previo, Fs
    t_actual = time.time()
    dt = t_actual - tiempo_previo
    tiempo_previo = t_actual
    if dt > 0: 
        tiempos_fps.append(dt)
        
    if len(tiempos_fps) > 50: 
        tiempos_fps.pop(0)

    if len(tiempos_fps) == 50:
        Fs = 1 / (sum(tiempos_fps) / 50)
        actualizar_coeficientes()
    #print(f"FPS: {Fs:.2f}", end="\r")
    
def procesar_loop():
    frecuencia_muestreo()
    
    ret, frame = cap.read()
    if not ret: 
        return

    coord_mano = Deteccion_mano_openCV(frame)

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0] # hl: hand landmark (puntos de referencia)
        
        BI,BD,mouse_X,mouse_Y,scroll,estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll, estado_scroll, sensibilidad,mano)
     

# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
        
    boton_inicio.setText("Iniciar")

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

# ==============================
# SETUP DE PYQT5 (SIN CLASE)
# ==============================
# EVITA EL ERROR DE "QApplication instance already exists"
app = QApplication.instance() or QApplication(sys.argv)

ventana = QWidget()
ventana.setWindowTitle("Control Gestual")
ventana.setFixedSize(500, 450)
ventana.closeEvent = cerrar_aplicacion # Inyectamos el cierre

layout = QVBoxLayout()
lbl_logo = QLabel()
pix = QPixmap("mouse virtual logo.jpg")
if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
lbl_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.clicked.connect(iniciar_programa)

# Mano
radio_der = QRadioButton("Derecha")
radio_der.setChecked(True)
radio_der.toggled.connect(lambda: globals().update(mano=0 if radio_der.isChecked() else 1))

radio_izq = QRadioButton("Izquierda")

# Sliders
sld_sens = QSlider(Qt.Horizontal); sld_sens.setRange(1, 10)
sld_sens.setValue(sensibilidad)
sld_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))

sld_fc = QSlider(Qt.Horizontal)
sld_fc.setRange(1, 10)
sld_fc.setValue(fc)
sld_fc.valueChanged.connect(lambda v: (globals().update(fc=v), actualizar_coeficientes()))

layout.addWidget(lbl_logo)
layout.addWidget(boton_inicio)

layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)

layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(sld_sens)

layout.addWidget(QLabel("Filtro (FC)"))
layout.addWidget(sld_fc)

ventana.setLayout(layout)
actualizar_coeficientes()
timer.timeout.connect(procesar_loop)

ventana.show()
app.exec_()

0

In [3]:
import sys  # Permite interactuar con el sistema (argumentos, salida, etc.)
import cv2  # OpenCV para manejo de cámara e imágenes
import mediapipe as mp  # Librería para detección de manos
import numpy as np  # Operaciones matemáticas y arrays
import pyautogui  # Control del mouse y teclado
import keyboard  # Manejo de teclado (aunque no se usa acá)
import time  # Manejo de tiempo
import scipy.signal as sig  # Filtros digitales
from PyQt5.QtWidgets import *  # Componentes de interfaz gráfica
from PyQt5.QtCore import *  # Núcleo de Qt (timers, señales, etc.)
from PyQt5.QtGui import QPixmap  # Manejo de imágenes en la UI

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False  # Desactiva la protección que detiene el mouse al ir a una esquina

# Variables de Control
sensibilidad = 2  # Multiplicador de movimiento del mouse
mano = 0  # 0 = derecha, 1 = izquierda
fc = 4  # Frecuencia de corte del filtro
ejecutando = False  # Estado del programa

positions = []  # Buffer de posiciones para suavizado
window_size_filtro = 1000  # Tamaño máximo del buffer
Fs = 30.0  # Frecuencia de muestreo inicial
orden = 2  # Orden del filtro
b, a = None, None  # Coeficientes del filtro

# Estado de Clicks y Scroll
FFD = 0  # Estado click derecho
FFI = 0  # Estado click izquierdo
estado_scroll = 0  # Estado del scroll
valor_anterior_y = 0  # Valor previo del dedo para scroll
tiempo_previo = time.time()  # Tiempo anterior para cálculo de FPS
tiempos_fps = []  # Lista de tiempos para estimar FPS

# Inicialización MediaPipe
mp_hands = mp.solutions.hands  # Módulo de manos
hands = mp_hands.Hands(
    static_image_mode=False,  # Modo video
    max_num_hands=1,  # Solo una mano
    min_detection_confidence=0.7  # Confianza mínima
)

cap = cv2.VideoCapture()  # Objeto de captura de cámara
timer = QTimer()  # Timer de Qt para ejecutar loop

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================
def actualizar_coeficientes():
    global b, a
    fs_segura = max(Fs, fc * 2.1)  # Evita violar Nyquist
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')  # Filtro pasa bajos


def suavizar_pos(x, y):
    global positions
    positions.append((x, y))  # Agrega nueva posición

    if len(positions) > window_size_filtro:
        positions.pop(0)  # Mantiene tamaño máximo

    if len(positions) > orden * 2:  # Si hay suficientes datos
        xs, ys = zip(*positions)  # Separa coordenadas
        return sig.lfilter(b, a, xs)[-1], sig.lfilter(b, a, ys)[-1]  # Filtra y devuelve último valor

    return x, y  # Si no hay suficientes datos, devuelve original


def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]:  # Si estaba suelto y ahora presiona
        pyautogui.mouseDown(button=tecla)  # Presiona botón
        return 1

    elif FF_val == 1 and not boton[0]:  # Si estaba presionado y ahora no
        pyautogui.mouseUp(button=tecla)  # Suelta botón
        return 0

    if boton[1]:  # Si detecta doble click
        pyautogui.click(button=tecla, clicks=2)

    return FF_val  # Devuelve estado


def mouse_ubicacion(hl):
    mouse_x = int(hl.landmark[0].x * 640)  # Coordenada X de la mano
    mouse_y = int(hl.landmark[0].y * 480)  # Coordenada Y de la mano

    mouse_x, mouse_y = suavizar_pos(mouse_x, mouse_y)  # Suaviza

    return mouse_x, mouse_y


def mouse_gesto_operacion(hl):
    bi1 = hl.landmark[8].y > hl.landmark[5].y  # Índice doblado
    bi2 = hl.landmark[16].y > hl.landmark[13].y  # Anular doblado

    bd1 = hl.landmark[12].y > hl.landmark[9].y  # Medio doblado
    bd2 = hl.landmark[20].y > hl.landmark[17].y  # Meñique doblado

    return (bi1, bi2), (bd1, bd2)  # Retorna estados


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)  # Ajusta tamaño y espejo
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # Procesa imagen
    return coord_mano


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)  # Posición
    BI, BD = mouse_gesto_operacion(hl)  # Botones
    scroll, estado_scroll = mouse_gesto_operacion_scroll(hl)  # Scroll

    return BI, BD, mouse_x, mouse_y, scroll, estado_scroll


FFD = FFI = 0  # Inicialización

def mouse_virtual(mouse_X, mouse_Y, BD, BI, scroll, estado_scroll, sensibilidad, mano):
    global FFD, FFI

    sw, sh = pyautogui.size()  # Tamaño de pantalla

    y = int((mouse_Y - 240) * sensibilidad * sh / 480)  # Escalado eje Y

    if mano == 0:
        x = int((mouse_X - 320) * sensibilidad * sw / 640)  # Derecha
    else:
        x = int(mouse_X * sensibilidad * sw / 640)  # Izquierda

    try:
        pyautogui.moveTo(x, y, _pause=False)  # Mueve mouse
    except:
        pass

    pyautogui.scroll(100 * scroll)  # Scroll

    if estado_scroll == 0:  # Si no está en modo scroll
        FFD = f_tecla(FFD, BD, 'right')  # Click derecho
        FFI = f_tecla(FFI, BI, 'left')  # Click izquierdo


estado_scroll = 0
valor_anterior_y = 0

def mouse_gesto_operacion_scroll(hl):
    global estado_scroll, valor_anterior_y

    scroll = 0  # Valor de salida

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 30) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 50) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    if gesto_estado_subir:
        estado_scroll = 1  # Modo subir

    if gesto_estado_bajar:
        estado_scroll = 2  # Modo bajar

    if gesto_estado_reposo:
        estado_scroll = 0  # Reposo

    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 1  # Scroll arriba

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0

    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y
        scroll = -1  # Scroll abajo

    return scroll, estado_scroll


def frecuencia_muestreo():
    global tiempo_previo, Fs

    t_actual = time.time()
    dt = t_actual - tiempo_previo  # Tiempo entre frames
    tiempo_previo = t_actual

    if dt > 0:
        tiempos_fps.append(dt)

    if len(tiempos_fps) > 50:
        tiempos_fps.pop(0)

    if len(tiempos_fps) == 50:
        Fs = 1 / (sum(tiempos_fps) / 50)  # FPS promedio
        actualizar_coeficientes()


def procesar_loop():
    frecuencia_muestreo()  # Actualiza FPS

    ret, frame = cap.read()  # Lee cámara
    if not ret:
        return

    coord_mano = Deteccion_mano_openCV(frame)  # Detecta mano

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0]  # Toma la mano

        BI, BD, mouse_X, mouse_Y, scroll, estado_scroll = Deteccion_gestos(hl)

        mouse_virtual(mouse_X, mouse_Y, BD, BI, scroll, estado_scroll, sensibilidad, mano)


# ==============================
# UI
# ==============================
def apagar_recursos():  # Función para detener todo el sistema (cámara + loop)
    global ejecutando  # Indica que se va a modificar la variable global "ejecutando"
    ejecutando = False  # Marca que el programa ya no está en ejecución

    timer.stop()  # Detiene el QTimer → deja de llamarse procesar_loop()

    if cap.isOpened():  # Verifica si la cámara está abierta
        cap.release()  # Libera la cámara (muy importante para no bloquearla)

    boton_inicio.setText("Iniciar")  # Cambia el texto del botón a "Iniciar"


def iniciar_programa():  # Función que se ejecuta al apretar el botón
    global ejecutando, tiempo_previo, positions, tiempos_fps  
    # Permite modificar estas variables globales

    if not ejecutando:  # Si el programa NO está corriendo
        cap.open(0, cv2.CAP_DSHOW)  # Abre la cámara (dispositivo 0)

        if not cap.isOpened():  # Si falla al abrir la cámara
            return QMessageBox.critical(None, "Error", "Cámara no disponible")  
            # Muestra mensaje de error y sale de la función

        ejecutando = True  # Marca que ahora el sistema está activo

        boton_inicio.setText("Pausar")  # Cambia el texto del botón a "Pausar"

        timer.start(30)  # Inicia el QTimer → ejecuta procesar_loop cada 30 ms (~33 FPS)

        positions, tiempos_fps = [], []  # Reinicia buffers (posiciones y tiempos)

        tiempo_previo = time.time()  # Guarda el tiempo actual (para cálculo de FPS)

    else:  # Si el programa YA estaba corriendo
        apagar_recursos()  # Lo detiene (toggle iniciar/pausar)


def cerrar_aplicacion(event):  # Función que se ejecuta al cerrar la ventana
    apagar_recursos()  # Detiene el sistema (timer + cámara)

    hands.close()  # Libera recursos de MediaPipe (detección de manos)

    cv2.destroyAllWindows()  # Cierra cualquier ventana de OpenCV

    event.accept()  # Acepta el evento de cierre (permite cerrar la app)

# ==============================
# PYQT5
# ==============================
app = QApplication.instance() or QApplication(sys.argv)  # Crea la app Qt o reutiliza una existente (motor de la GUI)

ventana = QWidget()  # Crea la ventana principal (contenedor)
ventana.setWindowTitle("Control Gestual")  # Define el título de la ventana
ventana.setFixedSize(500, 450)  # Fija el tamaño de la ventana (no redimensionable)
ventana.closeEvent = cerrar_aplicacion  # Asigna función personalizada al cerrar la ventana

layout = QVBoxLayout()  # Crea un layout vertical (organiza widgets de arriba hacia abajo)

lbl_logo = QLabel()  # Crea una etiqueta (para mostrar imagen o texto)
pix = QPixmap("mouse virtual logo.jpg")  # Carga la imagen desde archivo

if not pix.isNull():  # Verifica que la imagen se haya cargado correctamente
    lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))  # Escala la imagen sin deformarla y la asigna al label

lbl_logo.setAlignment(Qt.AlignCenter)  # Centra la imagen dentro del QLabel

boton_inicio = QPushButton("Iniciar")  # Crea el botón "Iniciar"
boton_inicio.clicked.connect(iniciar_programa)  # Conecta el click del botón con la función iniciar_programa()

# Mano
radio_der = QRadioButton("Derecha")  # Crea radio button "Derecha"
radio_der.setChecked(True)  # Lo deja seleccionado por defecto
radio_der.toggled.connect(lambda: globals().update(mano=0 if radio_der.isChecked() else 1))  
# Cuando cambia su estado:
# si está seleccionado → mano = 0 (derecha)
# si no → mano = 1 (izquierda)

radio_izq = QRadioButton("Izquierda")  # Crea radio button "Izquierda"

# Sliders
sld_sens = QSlider(Qt.Horizontal)  # Crea slider horizontal para sensibilidad
sld_sens.setRange(1, 10)  # Define rango de valores (1 a 10)
sld_sens.setValue(sensibilidad)  # Valor inicial del slider
sld_sens.valueChanged.connect(lambda v: globals().update(sensibilidad=v))  
# Cada vez que cambia el slider, actualiza la variable global "sensibilidad"

sld_fc = QSlider(Qt.Horizontal)  # Crea slider horizontal para frecuencia de corte
sld_fc.setRange(1, 10)  # Define rango de valores (1 a 10)
sld_fc.setValue(fc)  # Valor inicial del slider
sld_fc.valueChanged.connect(lambda v: (globals().update(fc=v), actualizar_coeficientes()))  
# Cuando cambia:
# 1. actualiza la variable global fc
# 2. recalcula los coeficientes del filtro

layout.addWidget(lbl_logo)  # Agrega el logo al layout (arriba de todo)
layout.addWidget(boton_inicio)  # Agrega el botón debajo del logo

layout.addWidget(QLabel("Mano"))  # Agrega etiqueta de texto "Mano"
layout.addWidget(radio_der)  # Agrega radio button "Derecha"
layout.addWidget(radio_izq)  # Agrega radio button "Izquierda"

layout.addWidget(QLabel("Sensibilidad"))  # Agrega etiqueta "Sensibilidad"
layout.addWidget(sld_sens)  # Agrega slider de sensibilidad

layout.addWidget(QLabel("Filtro (FC)"))  # Agrega etiqueta "Filtro (FC)"
layout.addWidget(sld_fc)  # Agrega slider de frecuencia de corte

ventana.setLayout(layout)  # Asigna el layout a la ventana (sin esto no se verían los widgets)

actualizar_coeficientes()  # Inicializa los coeficientes del filtro antes de empezar

timer.timeout.connect(procesar_loop)  # Conecta el timer para que ejecute procesar_loop() periódicamente

ventana.show()  # Muestra la ventana en pantalla

app.exec_()  # Inicia el loop de eventos de Qt (mantiene viva la aplicación)

0

In [1]:
!python --version

Python 3.9.23


In [1]:
!pip show mediapipe

Name: mediapipe
Version: 0.10.21
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: c:\users\usuario\anaconda3\envs\mouse\lib\site-packages
Requires: absl-py, attrs, flatbuffers, jax, jaxlib, matplotlib, numpy, opencv-contrib-python, protobuf, sentencepiece, sounddevice
Required-by: 


In [2]:
!pip list

Package                   Version
------------------------- -----------
absl-py                   2.3.1
anyio                     4.7.0
argon2-cffi               21.3.0
argon2-cffi-bindings      21.2.0
asttokens                 3.0.0
async-lru                 2.0.4
attrs                     24.3.0
babel                     2.16.0
backcall                  0.2.0
beautifulsoup4            4.13.5
bleach                    6.2.0
brotlicffi                1.0.9.2
certifi                   2025.8.3
cffi                      1.17.1
charset-normalizer        3.3.2
colorama                  0.4.6
comm                      0.2.1
contourpy                 1.3.0
cycler                    0.12.1
debugpy                   1.8.11
decorator                 5.1.1
defusedxml                0.7.1
exceptiongroup            1.2.0
executing                 0.8.3
fastjsonschema            2.20.0
flatbuffers               25.2.10
fonttools                 4.59.2
h11                       0.16.0
httpcore      

In [13]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False

# Variables de Control
Sens0 = 2.4
deltaSens = 1/10
BarraSens = 1
mano_camara = 0
fc = 4
ejecutando = False
positions = []
window_size_filtro = 15
Fs = 100
orden = 3
b, a = None, None

# Estado de Clicks y Scroll
FFD = 0
FFI = 0
estado_scroll = 0
valor_anterior_y = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1, 
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================
def actualizar_coeficientes():
    global b, a
    fs_segura = max(Fs, fc * 2.1)
    b, a = sig.butter(orden, fc/(0.5 * fs_segura), btype='low')


def suavizar_pos(x, y):
    global positions
    positions.append((x, y))
    if len(positions) > window_size_filtro: 
        positions.pop(0)
        
    if len(positions) > orden * 2:
        xs, ys = zip(*positions)
        return sig.lfilter(b, a, xs)[-1], sig.lfilter(b, a, ys)[-1]
    return x, y


def f_tecla(FF_val, boton, tecla):
    if FF_val == 0 and boton[0]: 
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]: 
        pyautogui.mouseUp(button=tecla)
        return 0
        
    if boton[1]: 
        pyautogui.click(button=tecla, clicks=2)
        
    return FF_val


def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y

def mouse_gesto_operacion(hl):
    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y
    
    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    return (bi1, bi2),(bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return coord_mano


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


FFD=FFI=0
def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
   # Mouse Move

    global  FFD,FFI
    sw, sh = pyautogui.size()
#3. A ambos nos sucedió que el ajuste de la sensibilidad vuelve inmanejable el cursor si pasas del step 2 de la slider. 
#Fijense si les permite , en lugar de 1 a 10, ponerlo de 1 a 3 (o 1 a 2) y luego usar pasos intermedios. 
#Revisen si pueden probar con otra cámara u otro ajuste de FPS de la misma, para confirmar que esa diferencia no sea por temas de la cámara en sí.
    Sens0 = 3.4
    deltaSens = 2/10
    
    sensibilidad = Sens0 + deltaSens*BarraSens
    
    
    y = int((mouse_Y - 320) * sensibilidad * sh / 480)

    if mano == 0:
        x = int((mouse_X - 426) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')


estado_scroll = 0
valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    
    global estado_scroll, valor_anterior_y
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 50) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 50) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    # ESTADOS
    if(gesto_estado_subir): estado_scroll = 1
    if(gesto_estado_bajar): estado_scroll = 2
    if(gesto_estado_reposo): estado_scroll = 0


    # SALIDAS
    #SALIDA SUBIDA
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        #print("1")
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5  and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0


    #SALIDA BAJADA    
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll,estado_scroll




def feedback(frame, coord_mano):
    # Espejo aquí para que todo (dibujo y mouse) coincida
    frame = cv2.flip(frame, 1)
    # Ya no procesamos 'hands.process' aquí, usamos lo que ya se detectó
    if coord_mano and coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            # Dibujamos sobre el frame original
            mp_drawing.draw_landmarks(
                frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    # 5. Mostrar la ventana con el resultado
    cv2.imshow('Detección de Mano', frame)

def procesar_loop():  
    ret, frame = cap.read()
    if not ret: 
        return    
   
    # 1. Una sola detección para todo
    coord_mano = Deteccion_mano_openCV(frame)

    # 2. Lógica del Mouse
    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0]
        BI, BD, mouse_X, mouse_Y, scroll, estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X, mouse_Y, BD, BI, scroll, estado_scroll, sensibilidad, mano)

    # 3. Pasamos el frame y la detección ya hecha a la función de dibujo
    feedback(frame, coord_mano)



    
# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
        
    boton_inicio.setText("Iniciar")

def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(30)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()

    


    

def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    cv2.destroyAllWindows()
    event.accept()

#6. Sería bueno que haya una forma de desactivarlo (más allá de pasar a la ventana y cliquear el botoncito) con algún pulsado de tecla 
#o secuencia de teclado, tal vez. 
def detener_con_esc():
    global ejecutando
    if ejecutando:
        apagar_recursos()

keyboard.add_hotkey('esc', detener_con_esc)
# 

# ==============================
# SETUP DE PYQT5 (SIN CLASE)
# ==============================
# EVITA EL ERROR DE "QApplication instance already exists"
app = QApplication.instance() or QApplication(sys.argv)

ventana = QWidget()
ventana.setWindowTitle("Control Gestual")
ventana.setFixedSize(500, 450)
ventana.closeEvent = cerrar_aplicacion # Inyectamos el cierre

layout = QVBoxLayout()
lbl_logo = QLabel()
pix = QPixmap("mouse virtual logo.jpg")
if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
lbl_logo.setAlignment(Qt.AlignCenter)

boton_inicio = QPushButton("Iniciar")
boton_inicio.clicked.connect(iniciar_programa)

# Mano
radio_der = QRadioButton("Derecha")
radio_der.setChecked(True)
radio_der.toggled.connect(lambda: globals().update(mano=0 if radio_der.isChecked() else 1))

radio_izq = QRadioButton("Izquierda")

# Sliders
sld_sens = QSlider(Qt.Horizontal); sld_sens.setRange(1, 10)
sld_sens.setValue(BarraSens)
sld_sens.valueChanged.connect(lambda v: globals().update(BarraSens=v))

sld_fc = QSlider(Qt.Horizontal)
sld_fc.setRange(1, 10)
sld_fc.setValue(fc)
sld_fc.valueChanged.connect(lambda v: (globals().update(fc=v), actualizar_coeficientes()))

layout.addWidget(lbl_logo)
layout.addWidget(boton_inicio)

layout.addWidget(QLabel("Mano"))
layout.addWidget(radio_der)
layout.addWidget(radio_izq)

layout.addWidget(QLabel("Sensibilidad"))
layout.addWidget(sld_sens)

layout.addWidget(QLabel("Filtro (FC)"))
layout.addWidget(sld_fc)

ventana.setLayout(layout)
actualizar_coeficientes()
timer.timeout.connect(procesar_loop)

ventana.show()
app.exec_()

0

In [2]:
import sys
import cv2
import mediapipe as mp
import pyautogui
import keyboard
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap, QImage

# Colores en formato BGR (OpenCV)
ROJO = (0, 0, 255)
VERDE = (0, 255, 0)
AZUL = (255, 0, 0)
AMARILLO = (0, 255, 255)
VIOLETA = (255, 0, 255)
CIAN = (255, 255, 0)
NARANJA = (0, 165, 255)
MARRON = (19, 69, 139)

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0 # mas velocidad

color_puntos = color_lineas = VERDE

# Variables de Control
BarraSens = 1
mano_ventana = 0

fc = 4
ejecutando = False
positions = []
window_size_filtro = 15
Fs = 100
orden = 1


# Estado de Clicks y Scroll
FFD = FFI = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1,     
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================

import math

class OneEuroFilter:
    def __init__(self, min_cutoff=1.0, beta=0.0, d_cutoff=1.0):
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self.x_prev = None
        self.dx_prev = 0

    def __call__(self, x, dt=1.0/30):
        if self.x_prev is None:
            self.x_prev = x
            return x
        
        # Calcular velocidad y filtrar
        dx = (x - self.x_prev) / dt
        edx = self._low_pass(dx, self.dx_prev, self._alpha(dt, self.d_cutoff))
        self.dx_prev = edx
        
        # Corte adaptativo
        cutoff = self.min_cutoff + self.beta * abs(edx)
        alpha = self._alpha(dt, cutoff)
        
        # Filtrar posición
        x_filtered = self._low_pass(x, self.x_prev, alpha)
        self.x_prev = x_filtered
        return x_filtered

    def _alpha(self, dt, cutoff):
        tau = 1.0 / (2 * math.pi * cutoff)
        return 1.0 / (1.0 + tau / dt)

    def _low_pass(self, x, x_prev, alpha):
        return alpha * x + (1.0 - alpha) * x_prev

# Inicializar uno para X y otro para Y
# Configuración "Pesada" (Mucha estabilidad, más inercia)
filtro_x = OneEuroFilter(min_cutoff=0.05, beta=0.001) 
filtro_y = OneEuroFilter(min_cutoff=0.05, beta=0.001)


def suavizar_pos(x, y):
    x_filt = filtro_x(x)
    y_filt = filtro_y(y)
    return x_filt, y_filt


def f_tecla(FF_val, boton, tecla):
    global color_puntos, color_lineas

    if FF_val == 0 and boton[0]:             
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]:         
        pyautogui.mouseUp(button=tecla)      
        return 0

    elif boton[1] == 1:
        pyautogui.click(button=tecla, clicks=2)    
        
    return FF_val
    

def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y
    

bi2_ant = 0
def mouse_gesto_operacion(hl):   
    global bi2_ant

    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y

    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    # Detectar flanco 0 -> 1
    if bi2 == 1 and bi2_ant == 0:
        salida_bi2 = 1
    else:
        salida_bi2 = 0

    # Guardar estado anterior
    bi2_ant = bi2  

    return (bi1, salida_bi2), (bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))    
    return coord_mano, frame # Retornamos también el frame procesado


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
    global FFD,FFI, BarraSens
    sw, sh = pyautogui.size()

    Sens0 = 3.4
    deltaSens = 2/5
    
    sensibilidad = Sens0 + deltaSens*BarraSens   
    
    y = int((mouse_Y - 320) * sensibilidad * sh / 480)

    if mano_ventana == 0:
        x = int((mouse_X - 426) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')

    
estado_scroll = valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    global estado_scroll, valor_anterior_y  
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 70) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 70) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    if gesto_estado_subir:  estado_scroll = 1        
    if gesto_estado_bajar:  estado_scroll = 2       
    if gesto_estado_reposo: estado_scroll = 0
                
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y        
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0

    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll, estado_scroll


mano_camara = 0
def feedback(frame, coord_mano):
    global mano_camara
    
    if coord_mano and coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame, 
                hand_landmarks, 
                mp_hands.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_puntos, thickness=2, circle_radius=4
                ),
                connection_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_lineas, thickness=3
                )
            )
        label = coord_mano.multi_handedness[0].classification[0].label

        mano_camara = int(label == "Left")
            
    # Conversión para mostrar en QLabel de PyQt5
    alto, ancho, canales = frame.shape
    paso = canales * ancho
    q_img = QImage(frame.data, ancho, alto, paso, QImage.Format_BGR888)
    lbl_logo.setPixmap(QPixmap.fromImage(q_img).scaled(lbl_logo.width(), lbl_logo.height(), Qt.KeepAspectRatio))


def color_manos(BD, BI, scroll, estado_scroll):    
    global color_puntos, color_lineas

    if mano_camara != mano_ventana:
        actualizar_texto("CAMBIAR MANO")
    
    elif BD[0] == 1:
        color_puntos = color_lineas = AZUL  
        actualizar_texto("CLICK IZQUIERDO")

    elif BI[1] == 1:
        color_puntos = color_lineas = CIAN  
        actualizar_texto("DOBLE CLICK IZQUIERDO")

    elif BI[0] == 1:
        color_puntos = color_lineas = ROJO  
        actualizar_texto("CLICK DERECHO")

    elif scroll == 1:
        color_puntos = color_lineas = NARANJA  
        actualizar_texto("SUBIENDO")

    elif scroll == -1:
        color_puntos = color_lineas = VIOLETA 
        actualizar_texto("BAJANDO")

    elif estado_scroll == 1:
        color_puntos = color_lineas = MARRON  
        actualizar_texto("ESTADO SUBIR")

    elif estado_scroll == 2:
        color_puntos = color_lineas = AMARILLO  
        actualizar_texto("ESTADO BAJAR")

    else:
        color_puntos = color_lineas = VERDE  
        actualizar_texto("ESPERA")
        
          
def procesar_loop():  
    ret, frame = cap.read()
    if not ret: 
        return    
   
    coord_mano, frame_procesado = Deteccion_mano_openCV(frame)

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0]
        BI, BD, mouse_X, mouse_Y, scroll, estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X, mouse_Y, BD, BI, scroll, estado_scroll, BarraSens, mano_ventana)
        color_manos( BD, BI, scroll, estado_scroll)
    
    feedback(frame_procesado, coord_mano)


# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
    
    # Restaurar el logo original al apagar
    pix = QPixmap("mouse virtual logo.jpg")
    if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
    boton_inicio.setText("Iniciar")


def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(1)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()


def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    event.accept()


def detener_con_esc():
    if ejecutando:
        apagar_recursos()
keyboard.add_hotkey('esc', detener_con_esc)


def actualizar_texto(mensaje):
    lbl_estado.setText(mensaje)

# ==============================
# SETUP DE PYQT5
# ==============================
app = QApplication.instance() or QApplication(sys.argv)

ventana_ui = QWidget()
ventana_ui.setWindowTitle("Control Gestual")
ventana_ui.setFixedSize(550, 650)   # ← NUEVA ALTURA

ventana_ui.closeEvent = cerrar_aplicacion

# ==============================
# LAYOUT PRINCIPAL
# ==============================
layout = QVBoxLayout()
layout.setSpacing(10)
layout.setContentsMargins(12,12,12,12)

# ==============================
# LABEL VIDEO / LOGO
# ==============================
lbl_logo = QLabel()

pix = QPixmap("mouse virtual logo.jpg")

if not pix.isNull():
    lbl_logo.setPixmap(
        pix.scaled(
            180, 120,
            Qt.KeepAspectRatio
        )
    )

lbl_logo.setAlignment(Qt.AlignCenter)

# MÁS CHICO
lbl_logo.setMinimumSize(420, 260)

lbl_logo.setStyleSheet("""
background-color: #222;
border: 1px solid #444;
""")

# ==============================
# LABEL ESTADO
# ==============================
lbl_estado = QLabel("Esperando...")

lbl_estado.setAlignment(Qt.AlignCenter)

lbl_estado.setStyleSheet("""
font-size: 16px;
font-weight: bold;
color: #00FF00;
background-color: #111;
padding: 4px;
""")

# ==============================
# BOTÓN INICIO
# ==============================
boton_inicio = QPushButton("Iniciar")

# MÁS BAJO
boton_inicio.setFixedHeight(40)

boton_inicio.setStyleSheet("""
font-size: 16px;
font-weight: bold;
""")

boton_inicio.clicked.connect(
    iniciar_programa
)

# ==============================
# TOGGLE MANO
# ==============================
toggle_mano = QPushButton("MANO DERECHA")

toggle_mano.setCheckable(True)

# MÁS CHICO
toggle_mano.setFixedHeight(50)

toggle_mano.setStyleSheet("""

QPushButton {
    font-size: 18px;
    font-weight: bold;
    background-color: #1E88E5;
    color: white;
    border-radius: 10px;
    padding: 6px;
}

QPushButton:checked {
    background-color: #8E24AA;
}

""")

# FUNCIÓN TOGGLE
def cambiar_mano():

    global mano_ventana

    if toggle_mano.isChecked():

        mano_ventana = 1
        toggle_mano.setText("MANO IZQUIERDA")

    else:

        mano_ventana = 0
        toggle_mano.setText("MANO DERECHA")

toggle_mano.clicked.connect(
    cambiar_mano
)

# ==============================
# SLIDER SENSIBILIDAD
# ==============================
sld_sens = QSlider(Qt.Horizontal)

sld_sens.setRange(1, 5)
sld_sens.setValue(BarraSens)

sld_sens.setStyleSheet("""

QSlider::groove:horizontal {
    background: #444;
    height: 10px;
    border-radius: 5px;
}

QSlider::handle:horizontal {
    background: #00FFAA;
    width: 28px;
    margin: -6px 0;
    border-radius: 14px;
}

""")

sld_sens.valueChanged.connect(
    lambda v: globals().update(
        BarraSens=v
    )
)

# ==============================
# LABELS CONFIG
# ==============================
lbl_sens = QLabel("Sensibilidad")

lbl_sens.setStyleSheet("""
font-size: 16px;
font-weight: bold;
""")

lbl_cam = QLabel("Vista de Cámara:")

# MÁS CHICO
lbl_cam.setStyleSheet("""
font-size: 24px;
font-weight: bold;
""")

# ==============================
# BOTÓN HELP
# ==============================
boton_help = QPushButton("HELP")

# MÁS BAJO
boton_help.setFixedHeight(50)

boton_help.setStyleSheet("""

QPushButton {
    font-size: 18px;
    font-weight: bold;
    background-color: #455A64;
    color: white;
    border-radius: 10px;
    padding: 6px;
}

QPushButton:hover {
    background-color: #546E7A;
}

""")

# FUNCIÓN ABRIR PDF
def abrir_help():
    import os

    ruta_pdf = "manual.pdf"   # nombre del PDF
    if os.path.exists(ruta_pdf):
        os.startfile(ruta_pdf)

    # Abrir video
    ruta_video = "tutorial.mp4"
    if os.path.exists(ruta_video):
        os.startfile(ruta_video)

boton_help.clicked.connect(
    abrir_help
)

# ==============================
# ORGANIZACIÓN LAYOUT
# ==============================
layout.addWidget(boton_inicio)
layout.addSpacing(10)
layout.addWidget(boton_help)
layout.addSpacing(15)
layout.addWidget(toggle_mano)
layout.addSpacing(15)
layout.addWidget(lbl_sens)
layout.addWidget(sld_sens)
layout.addSpacing(15)
layout.addWidget(lbl_cam)
layout.addWidget(lbl_logo)
layout.addWidget(lbl_estado)

# ==============================
# CONFIG FINAL
# ==============================
ventana_ui.setLayout(layout)

timer.timeout.connect(
    procesar_loop
)

ventana_ui.show()
app.exec_()

0

In [ ]:
# Inicialización MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1, 
    model_complexity=0, # Más rápido
    min_detection_confidence=0.7
)

In [2]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap, QImage

# Colores en formato BGR (OpenCV)
ROJO = (0, 0, 255)
VERDE = (0, 255, 0)
AZUL = (255, 0, 0)
AMARILLO = (0, 255, 255)
VIOLETA = (255, 0, 255)
CIAN = (255, 255, 0)
NARANJA = (0, 165, 255)
MARRON = (19, 69, 139)

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0 # mas velocidad

color_puntos = color_lineas = VERDE

# Variables de Control
BarraSens = 1
mano_ventana = 0


ejecutando = False
positions = []


# Estado de Clicks y Scroll
FFD = FFI = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1,     
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================

import math

class OneEuroFilter:
    def __init__(self, min_cutoff=1.0, beta=0.0, d_cutoff=1.0):
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self.x_prev = None
        self.dx_prev = 0

    def __call__(self, x, dt=1.0/30):
        if self.x_prev is None:
            self.x_prev = x
            return x
        
        # Calcular velocidad y filtrar
        dx = (x - self.x_prev) / dt
        edx = self._low_pass(dx, self.dx_prev, self._alpha(dt, self.d_cutoff))
        self.dx_prev = edx
        
        # Corte adaptativo
        cutoff = self.min_cutoff + self.beta * abs(edx)
        alpha = self._alpha(dt, cutoff)
        
        # Filtrar posición
        x_filtered = self._low_pass(x, self.x_prev, alpha)
        self.x_prev = x_filtered
        return x_filtered

    def _alpha(self, dt, cutoff):
        tau = 1.0 / (2 * math.pi * cutoff)
        return 1.0 / (1.0 + tau / dt)

    def _low_pass(self, x, x_prev, alpha):
        return alpha * x + (1.0 - alpha) * x_prev

# Inicializar uno para X y otro para Y
# Configuración "Pesada" (Mucha estabilidad, más inercia)
filtro_x = OneEuroFilter(min_cutoff=0.05, beta=0.001) 
filtro_y = OneEuroFilter(min_cutoff=0.05, beta=0.001)


def suavizar_pos(x, y):
    x_filt = filtro_x(x)
    y_filt = filtro_y(y)
    return x_filt, y_filt


def f_tecla(FF_val, boton, tecla):
    global color_puntos, color_lineas

    if FF_val == 0 and boton[0]:             
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]:         
        pyautogui.mouseUp(button=tecla)      
        return 0

    elif boton[1] == 1:
        pyautogui.click(button=tecla, clicks=2)    
        
    return FF_val
    

def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y
    

bi2_ant = 0
def mouse_gesto_operacion(hl):   
    global bi2_ant

    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y

    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    # Detectar flanco 0 -> 1
    if bi2 == 1 and bi2_ant == 0:
        salida_bi2 = 1
    else:
        salida_bi2 = 0

    # Guardar estado anterior
    bi2_ant = bi2  

    return (bi1, salida_bi2), (bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))    
    return coord_mano, frame 


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
    global FFD,FFI, BarraSens
    sw, sh = pyautogui.size()

    Sens0 = 3.4
    deltaSens = 2/5
    
    sensibilidad = Sens0 + deltaSens*BarraSens   
    
    y = int((mouse_Y - 320) * sensibilidad * sh / 480)

    if mano_ventana == 0:
        x = int((mouse_X - 426) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')

    
estado_scroll = valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    global estado_scroll, valor_anterior_y  
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 70) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 70) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    if gesto_estado_subir:  estado_scroll = 1        
    if gesto_estado_bajar:  estado_scroll = 2       
    if gesto_estado_reposo: estado_scroll = 0
                
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y        
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0

    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll, estado_scroll


mano_camara = 0
def feedback(frame, coord_mano):
    global mano_camara
    
    if coord_mano and coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame, 
                hand_landmarks, 
                mp_hands.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_puntos, thickness=2, circle_radius=4
                ),
                connection_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_lineas, thickness=3
                )
            )
        label = coord_mano.multi_handedness[0].classification[0].label

        mano_camara = int(label == "Left")
            
    # Conversión para mostrar en QLabel de PyQt5
    alto, ancho, canales = frame.shape
    paso = canales * ancho
    q_img = QImage(frame.data, ancho, alto, paso, QImage.Format_BGR888)
    lbl_logo.setPixmap(QPixmap.fromImage(q_img).scaled(lbl_logo.width(), lbl_logo.height(), Qt.KeepAspectRatio))


def color_manos(BD, BI, scroll, estado_scroll):    
    global color_puntos, color_lineas

    if mano_camara != mano_ventana:
        actualizar_texto("CAMBIAR MANO")
    
    elif BD[0] == 1:
        color_puntos = color_lineas = AZUL  
        actualizar_texto("CLICK DERECHO")

    elif BI[1] == 1:
        color_puntos = color_lineas = CIAN  
        actualizar_texto("DOBLE CLICK IZQUIERDO")

    elif BI[0] == 1:
        color_puntos = color_lineas = ROJO  
        actualizar_texto("CLICK IZQUIERDO")

    elif scroll == 1:
        color_puntos = color_lineas = NARANJA  
        actualizar_texto("SUBIENDO")

    elif scroll == -1:
        color_puntos = color_lineas = VIOLETA 
        actualizar_texto("BAJANDO")

    elif estado_scroll == 1:
        color_puntos = color_lineas = MARRON  
        actualizar_texto("ESTADO SUBIR")

    elif estado_scroll == 2:
        color_puntos = color_lineas = AMARILLO  
        actualizar_texto("ESTADO BAJAR")

    else:
        color_puntos = color_lineas = VERDE  
        actualizar_texto("ESPERA")
        
          
def procesar_loop():  
    ret, frame = cap.read()
    if not ret: 
        return    
   
    coord_mano, frame_procesado = Deteccion_mano_openCV(frame)

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0]
        BI, BD, mouse_X, mouse_Y, scroll, estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X, mouse_Y, BD, BI, scroll, estado_scroll, BarraSens, mano_ventana)
        color_manos( BD, BI, scroll, estado_scroll)
    
    feedback(frame_procesado, coord_mano)


# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
    
    # Restaurar el logo original al apagar
    pix = QPixmap("mouse virtual logo.jpg")
    if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
    boton_inicio.setText("Iniciar")


def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(1)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()


def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    event.accept()


def detener_con_esc():
    if ejecutando:
        apagar_recursos()



def actualizar_texto(mensaje):
    lbl_estado.setText(mensaje)

# ==============================
# SETUP DE PYQT5
# ==============================
app = QApplication.instance() or QApplication(sys.argv)

ventana_ui = QWidget()
ventana_ui.setWindowTitle("Control Gestual")
ventana_ui.setFixedSize(550, 650) 

ventana_ui.closeEvent = cerrar_aplicacion

# CAPTURAR ESC
def keyPressEvent(event):
    if event.key() == Qt.Key_Escape:
        detener_con_esc()

ventana_ui.keyPressEvent = keyPressEvent

# ==============================
# LAYOUT PRINCIPAL
# ==============================
layout = QVBoxLayout()
layout.setSpacing(10)
layout.setContentsMargins(12,12,12,12)

# ==============================
# LABEL VIDEO / LOGO
# ==============================
lbl_logo = QLabel()

pix = QPixmap("mouse virtual logo.jpg")

if not pix.isNull():
    lbl_logo.setPixmap(
        pix.scaled(
            180, 120,
            Qt.KeepAspectRatio
        )
    )

lbl_logo.setAlignment(Qt.AlignCenter)
lbl_logo.setMinimumSize(420, 260)
lbl_logo.setStyleSheet("""
background-color: #222;
border: 1px solid #444;
""")

# ==============================
# LABEL ESTADO
# ==============================
lbl_estado = QLabel("Esperando...")
lbl_estado.setAlignment(Qt.AlignCenter)
lbl_estado.setStyleSheet("""
font-size: 16px;
font-weight: bold;
color: #00FF00;
background-color: #111;
padding: 4px;
""")

# ==============================
# BOTÓN INICIO
# ==============================
boton_inicio = QPushButton("Iniciar")
boton_inicio.setFixedHeight(40)
boton_inicio.setStyleSheet("""
font-size: 16px;
font-weight: bold;
""")

boton_inicio.clicked.connect(
    iniciar_programa
)

# ==============================
# TOGGLE MANO
# ==============================
toggle_mano = QPushButton("MANO DERECHA")
toggle_mano.setCheckable(True)
toggle_mano.setFixedHeight(50)
toggle_mano.setStyleSheet("""

QPushButton {
    font-size: 18px;
    font-weight: bold;
    background-color: #1E88E5;
    color: white;
    border-radius: 10px;
    padding: 6px;
}

QPushButton:checked {
    background-color: #8E24AA;
}

""")

# FUNCIÓN TOGGLE
def cambiar_mano():

    global mano_ventana

    if toggle_mano.isChecked():

        mano_ventana = 1
        toggle_mano.setText("MANO IZQUIERDA")

    else:

        mano_ventana = 0
        toggle_mano.setText("MANO DERECHA")

toggle_mano.clicked.connect(
    cambiar_mano
)

# ==============================
# SLIDER SENSIBILIDAD
# ==============================
sld_sens = QSlider(Qt.Horizontal)

sld_sens.setRange(1, 5)
sld_sens.setValue(BarraSens)

sld_sens.setStyleSheet("""

QSlider::groove:horizontal {
    background: #444;
    height: 10px;
    border-radius: 5px;
}

QSlider::handle:horizontal {
    background: #00FFAA;
    width: 28px;
    margin: -6px 0;
    border-radius: 14px;
}

""")

sld_sens.valueChanged.connect(
    lambda v: globals().update(
        BarraSens=v
    )
)

# ==============================
# LABELS CONFIG
# ==============================
lbl_sens = QLabel("Sensibilidad")
lbl_sens.setStyleSheet("""
font-size: 16px;
font-weight: bold;
""")

lbl_cam = QLabel("Vista de Cámara:")
lbl_cam.setStyleSheet("""
font-size: 24px;
font-weight: bold;
""")

# ==============================
# BOTÓN HELP
# ==============================
boton_help = QPushButton("HELP")

# MÁS BAJO
boton_help.setFixedHeight(50)

boton_help.setStyleSheet("""

QPushButton {
    font-size: 18px;
    font-weight: bold;
    background-color: #455A64;
    color: white;
    border-radius: 10px;
    padding: 6px;
}

QPushButton:hover {
    background-color: #546E7A;
}

""")

# FUNCIÓN ABRIR PDF
def abrir_help():
    import os

    ruta_pdf = "manual.pdf" 
    if os.path.exists(ruta_pdf):
        os.startfile(ruta_pdf)

    ruta_video = "tutorial.mp4"
    if os.path.exists(ruta_video):
        os.startfile(ruta_video)

boton_help.clicked.connect(
    abrir_help
)

# ==============================
# ORGANIZACIÓN LAYOUT
# ==============================
layout.addWidget(boton_inicio)
layout.addSpacing(10)
layout.addWidget(boton_help)
layout.addSpacing(15)
layout.addWidget(toggle_mano)
layout.addSpacing(15)
layout.addWidget(lbl_sens)
layout.addWidget(sld_sens)
layout.addSpacing(15)
layout.addWidget(lbl_cam)
layout.addWidget(lbl_logo)
layout.addWidget(lbl_estado)

# ==============================
# CONFIG FINAL
# ==============================
ventana_ui.setLayout(layout)

timer.timeout.connect(
    procesar_loop
)

ventana_ui.show()
app.exec_()

0

In [5]:
import sys
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import keyboard
import time
import scipy.signal as sig
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import QPixmap, QImage

# Colores en formato BGR (OpenCV)
ROJO = (0, 0, 255)
VERDE = (0, 255, 0)
AZUL = (255, 0, 0)
AMARILLO = (0, 255, 255)
VIOLETA = (255, 0, 255)
CIAN = (255, 255, 0)
NARANJA = (0, 165, 255)
MARRON = (19, 69, 139)

# ==============================
# CONFIGURACIÓN GLOBAL
# ==============================
pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0 # mas velocidad

color_puntos = color_lineas = VERDE

# Variables de Control
BarraSens = 1
mano_ventana = 0

fc = 4
ejecutando = False
positions = []
window_size_filtro = 15
Fs = 100
orden = 1


# Estado de Clicks y Scroll
FFD = FFI = 0
tiempo_previo = time.time()
tiempos_fps = []

# Inicialización MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=1,     
    min_detection_confidence=0.7
)
cap = cv2.VideoCapture()
timer = QTimer()

# ==============================
# LÓGICA DE PROCESAMIENTO
# ==============================

import math

class OneEuroFilter:
    def __init__(self, min_cutoff=1.0, beta=0.0, d_cutoff=1.0):
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self.x_prev = None
        self.dx_prev = 0

    def __call__(self, x, dt=1.0/30):
        if self.x_prev is None:
            self.x_prev = x
            return x
        
        # Calcular velocidad y filtrar
        dx = (x - self.x_prev) / dt
        edx = self._low_pass(dx, self.dx_prev, self._alpha(dt, self.d_cutoff))
        self.dx_prev = edx
        
        # Corte adaptativo
        cutoff = self.min_cutoff + self.beta * abs(edx)
        alpha = self._alpha(dt, cutoff)
        
        # Filtrar posición
        x_filtered = self._low_pass(x, self.x_prev, alpha)
        self.x_prev = x_filtered
        return x_filtered

    def _alpha(self, dt, cutoff):
        tau = 1.0 / (2 * math.pi * cutoff)
        return 1.0 / (1.0 + tau / dt)

    def _low_pass(self, x, x_prev, alpha):
        return alpha * x + (1.0 - alpha) * x_prev

# Inicializar uno para X y otro para Y
# Configuración "Pesada" (Mucha estabilidad, más inercia)
filtro_x = OneEuroFilter(min_cutoff=0.05, beta=0.001) 
filtro_y = OneEuroFilter(min_cutoff=0.05, beta=0.001)


def suavizar_pos(x, y):
    x_filt = filtro_x(x)
    y_filt = filtro_y(y)
    return x_filt, y_filt


def f_tecla(FF_val, boton, tecla):
    global color_puntos, color_lineas

    if FF_val == 0 and boton[0]:             
        pyautogui.mouseDown(button=tecla)
        return 1
        
    elif FF_val == 1 and not boton[0]:         
        pyautogui.mouseUp(button=tecla)      
        return 0

    elif boton[1] == 1:
        pyautogui.click(button=tecla, clicks=2)    
        
    return FF_val
    

def mouse_ubicacion(hl):
    mouse_x, mouse_y = int(hl.landmark[0].x * 640), int(hl.landmark[0].y * 480)
    mouse_x, mouse_y = suavizar_pos((mouse_x), (mouse_y))
    
    return mouse_x, mouse_y
    

bi2_ant = 0
def mouse_gesto_operacion(hl):   
    global bi2_ant

    bi1 = hl.landmark[8].y > hl.landmark[5].y    
    bi2 = hl.landmark[16].y > hl.landmark[13].y

    bd1 = hl.landmark[12].y > hl.landmark[9].y
    bd2 = hl.landmark[20].y > hl.landmark[17].y

    # Detectar flanco 0 -> 1
    if bi2 == 1 and bi2_ant == 0:
        salida_bi2 = 1
    else:
        salida_bi2 = 0

    # Guardar estado anterior
    bi2_ant = bi2  

    return (bi1, salida_bi2), (bd1, bd2)


def Deteccion_mano_openCV(frame):
    frame = cv2.flip(cv2.resize(frame, (640, 480)), 1)
    coord_mano = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))    
    return coord_mano, frame # Retornamos también el frame procesado


def Deteccion_gestos(hl):
    mouse_x, mouse_y = mouse_ubicacion(hl)
    BI,BD = mouse_gesto_operacion(hl)
    scroll,estado_scroll = mouse_gesto_operacion_scroll(hl)
    
    return BI,BD,mouse_x, mouse_y,scroll,estado_scroll


def mouse_virtual(mouse_X ,mouse_Y , BD, BI,scroll,estado_scroll, sensibilidad,mano):
    global FFD,FFI, BarraSens
    sw, sh = pyautogui.size()

    Sens0 = 3.4
    deltaSens = 2/5
    
    sensibilidad = Sens0 + deltaSens*BarraSens   
    
    y = int((mouse_Y - 320) * sensibilidad * sh / 480)

    if mano_ventana == 0:
        x = int((mouse_X - 426) * sensibilidad * sw / 640) 
    else:
        x = int(mouse_X * sensibilidad * sw / 640)

    try: pyautogui.moveTo(x, y, _pause=False)
    except: pass

    pyautogui.scroll(100*scroll)

    if estado_scroll == 0:
        FFD = f_tecla(FFD, BD, 'right')
        FFI = f_tecla(FFI, BI, 'left')

    
estado_scroll = valor_anterior_y = 0
def mouse_gesto_operacion_scroll(hl):
    global estado_scroll, valor_anterior_y  
    scroll = 0

    gesto_estado_subir = (abs(hl.landmark[8].x - hl.landmark[13].x)*640 < 70) and estado_scroll == 0
    gesto_estado_bajar = (abs(hl.landmark[5].y - hl.landmark[8].y)*480 < 70) and estado_scroll == 0
    gesto_estado_reposo = (abs(hl.landmark[0].y - hl.landmark[9].y)/5) < abs(hl.landmark[17].x - hl.landmark[5].x)

    if gesto_estado_subir:  estado_scroll = 1        
    if gesto_estado_bajar:  estado_scroll = 2       
    if gesto_estado_reposo: estado_scroll = 0
                
    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y        
        scroll = 1

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 1:
        valor_anterior_y = hl.landmark[8].y
        scroll = 0

    if (hl.landmark[8].y - valor_anterior_y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y              
        scroll = 0

    if (valor_anterior_y - hl.landmark[8].y)*480 > 5 and estado_scroll == 2:
        valor_anterior_y = hl.landmark[8].y                
        scroll = -1
        
    return scroll, estado_scroll


mano_camara = 0
def feedback(frame, coord_mano):
    global mano_camara
    
    if coord_mano and coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame, 
                hand_landmarks, 
                mp_hands.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_puntos, thickness=2, circle_radius=4
                ),
                connection_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_lineas, thickness=3
                )
            )
        label = coord_mano.multi_handedness[0].classification[0].label

        mano_camara = int(label == "Left")
            
    # Conversión para mostrar en QLabel de PyQt5
    alto, ancho, canales = frame.shape
    paso = canales * ancho
    q_img = QImage(frame.data, ancho, alto, paso, QImage.Format_BGR888)
    lbl_logo.setPixmap(QPixmap.fromImage(q_img).scaled(lbl_logo.width(), lbl_logo.height(), Qt.KeepAspectRatio))


def color_manos(BD, BI, scroll, estado_scroll):    
    global color_puntos, color_lineas

    if mano_camara != mano_ventana:
        actualizar_texto("CAMBIAR MANO")
    
    elif BD[0] == 1:
        color_puntos = color_lineas = AZUL  
        actualizar_texto("CLICK DERECHO")

    elif BI[1] == 1:
        color_puntos = color_lineas = CIAN  
        actualizar_texto("DOBLE CLICK IZQUIERDO")

    elif BI[0] == 1:
        color_puntos = color_lineas = ROJO  
        actualizar_texto("CLICK IZQUIERDO")

    elif scroll == 1:
        color_puntos = color_lineas = NARANJA  
        actualizar_texto("SUBIENDO")

    elif scroll == -1:
        color_puntos = color_lineas = VIOLETA 
        actualizar_texto("BAJANDO")

    elif estado_scroll == 1:
        color_puntos = color_lineas = MARRON  
        actualizar_texto("ESTADO SUBIR")

    elif estado_scroll == 2:
        color_puntos = color_lineas = AMARILLO  
        actualizar_texto("ESTADO BAJAR")

    else:
        color_puntos = color_lineas = VERDE  
        actualizar_texto("ESPERA")
        
          
def procesar_loop():  
    ret, frame = cap.read()
    if not ret: 
        return    
   
    coord_mano, frame_procesado = Deteccion_mano_openCV(frame)

    if coord_mano.multi_hand_landmarks:
        hl = coord_mano.multi_hand_landmarks[0]
        BI, BD, mouse_X, mouse_Y, scroll, estado_scroll = Deteccion_gestos(hl)      
        mouse_virtual(mouse_X, mouse_Y, BD, BI, scroll, estado_scroll, BarraSens, mano_ventana)
        color_manos( BD, BI, scroll, estado_scroll)
    
    feedback(frame_procesado, coord_mano)


# ==============================
# CONTROL DE UI Y RECURSOS
# ==============================
def apagar_recursos():
    global ejecutando
    ejecutando = False
    timer.stop()
    if cap.isOpened(): 
        cap.release()
    
    # Restaurar el logo original al apagar
    pix = QPixmap("mouse virtual logo.jpg")
    if not pix.isNull(): lbl_logo.setPixmap(pix.scaled(200, 150, Qt.KeepAspectRatio))
    boton_inicio.setText("Iniciar")


def iniciar_programa():
    global ejecutando, tiempo_previo, positions, tiempos_fps
    if not ejecutando:
        cap.open(0, cv2.CAP_DSHOW)
        if not cap.isOpened():
            return QMessageBox.critical(None, "Error", "Cámara no disponible")
        ejecutando = True
        boton_inicio.setText("Pausar")
        timer.start(1)
        positions, tiempos_fps = [], []
        tiempo_previo = time.time()
    else:
        apagar_recursos()


def cerrar_aplicacion(event):
    apagar_recursos()
    hands.close()
    event.accept()


def detener_con_esc():
    if ejecutando:
        apagar_recursos()



def actualizar_texto(mensaje):
    lbl_estado.setText(mensaje)

# ==============================
# SETUP DE PYQT5
# ==============================
app = QApplication.instance() or QApplication(sys.argv)

ventana_ui = QWidget()
ventana_ui.setWindowTitle("Control Gestual")
ventana_ui.setFixedSize(550, 650)   # ← NUEVA ALTURA

ventana_ui.closeEvent = cerrar_aplicacion

# CAPTURAR ESC
def keyPressEvent(event):
    if event.key() == Qt.Key_Escape:
        detener_con_esc()

ventana_ui.keyPressEvent = keyPressEvent

# ==============================
# LAYOUT PRINCIPAL
# ==============================
layout = QVBoxLayout()
layout.setSpacing(10)
layout.setContentsMargins(12,12,12,12)

# ==============================
# LABEL VIDEO / LOGO
# ==============================
lbl_logo = QLabel()

pix = QPixmap("mouse virtual logo.jpg")

if not pix.isNull():
    lbl_logo.setPixmap(
        pix.scaled(
            180, 120,
            Qt.KeepAspectRatio
        )
    )

lbl_logo.setAlignment(Qt.AlignCenter)

# MÁS CHICO
lbl_logo.setMinimumSize(420, 260)

lbl_logo.setStyleSheet("""
background-color: #222;
border: 1px solid #444;
""")

# ==============================
# LABEL ESTADO
# ==============================
lbl_estado = QLabel("Esperando...")

lbl_estado.setAlignment(Qt.AlignCenter)

lbl_estado.setStyleSheet("""
font-size: 16px;
font-weight: bold;
color: #00FF00;
background-color: #111;
padding: 4px;
""")

# ==============================
# BOTÓN INICIO
# ==============================
boton_inicio = QPushButton("Iniciar")

# MÁS BAJO
boton_inicio.setFixedHeight(40)

boton_inicio.setStyleSheet("""
font-size: 16px;
font-weight: bold;
""")

boton_inicio.clicked.connect(
    iniciar_programa
)

# ==============================
# TOGGLE MANO
# ==============================
toggle_mano = QPushButton("MANO DERECHA")

toggle_mano.setCheckable(True)

# MÁS CHICO
toggle_mano.setFixedHeight(50)

toggle_mano.setStyleSheet("""

QPushButton {
    font-size: 18px;
    font-weight: bold;
    background-color: #1E88E5;
    color: white;
    border-radius: 10px;
    padding: 6px;
}

QPushButton:checked {
    background-color: #8E24AA;
}

""")

# FUNCIÓN TOGGLE
def cambiar_mano():

    global mano_ventana

    if toggle_mano.isChecked():

        mano_ventana = 1
        toggle_mano.setText("MANO IZQUIERDA")

    else:

        mano_ventana = 0
        toggle_mano.setText("MANO DERECHA")

toggle_mano.clicked.connect(
    cambiar_mano
)

# ==============================
# SLIDER SENSIBILIDAD
# ==============================
sld_sens = QSlider(Qt.Horizontal)

sld_sens.setRange(1, 5)
sld_sens.setValue(BarraSens)

sld_sens.setStyleSheet("""

QSlider::groove:horizontal {
    background: #444;
    height: 10px;
    border-radius: 5px;
}

QSlider::handle:horizontal {
    background: #00FFAA;
    width: 28px;
    margin: -6px 0;
    border-radius: 14px;
}

""")

sld_sens.valueChanged.connect(
    lambda v: globals().update(
        BarraSens=v
    )
)

# ==============================
# LABELS CONFIG
# ==============================
lbl_sens = QLabel("Sensibilidad")

lbl_sens.setStyleSheet("""
font-size: 16px;
font-weight: bold;
""")

lbl_cam = QLabel("Vista de Cámara:")

# MÁS CHICO
lbl_cam.setStyleSheet("""
font-size: 24px;
font-weight: bold;
""")

# ==============================
# BOTÓN HELP
# ==============================
boton_help = QPushButton("HELP")

# MÁS BAJO
boton_help.setFixedHeight(50)

boton_help.setStyleSheet("""

QPushButton {
    font-size: 18px;
    font-weight: bold;
    background-color: #455A64;
    color: white;
    border-radius: 10px;
    padding: 6px;
}

QPushButton:hover {
    background-color: #546E7A;
}

""")

# FUNCIÓN ABRIR PDF
def abrir_help():
    import os

    ruta_pdf = "manual.pdf"   # nombre del PDF
    if os.path.exists(ruta_pdf):
        os.startfile(ruta_pdf)

    # Abrir video
    ruta_video = "tutorial.mp4"
    if os.path.exists(ruta_video):
        os.startfile(ruta_video)

boton_help.clicked.connect(
    abrir_help
)

# ==============================
# ORGANIZACIÓN LAYOUT
# ==============================
layout.addWidget(boton_inicio)
layout.addSpacing(10)
layout.addWidget(boton_help)
layout.addSpacing(15)
layout.addWidget(toggle_mano)
layout.addSpacing(15)
layout.addWidget(lbl_sens)
layout.addWidget(sld_sens)
layout.addSpacing(15)
layout.addWidget(lbl_cam)
layout.addWidget(lbl_logo)
layout.addWidget(lbl_estado)

# ==============================
# CONFIG FINAL
# ==============================
ventana_ui.setLayout(layout)

timer.timeout.connect(
    procesar_loop
)

ventana_ui.show()
app.exec_()

0

In [6]:
!pip freeze > requirements.txt